# Sprint 2

## python
python3 -m venv .venv
source .venv/bin/activate
python3 -m pip install --upgrade pip

# Этап 1. Аудит и подготовка корпуса

## pip
```
pip install ipykernel 
pip install tqdm ipywidgets
pip install ftfy bs4 trafilatura
python -m pip install urllib3[socks]
```

## dataset
wget https://code.s3.yandex.net/ai-engineer/Итоговый_проект_спринт2/filmoskop_kb.json -O filmoskop_kb.json

In [70]:
'''
1.0 Load primary docs
'''
import json

with open("filmoskop_kb.json", encoding="utf-8") as f:
    data = json.load(f)
docs = data["docs"]        # 181 раздел базы знаний
queries = data["queries"]  # 66 тестовых запросов с разметкой

print("Разделов:", len(docs), "| Запросов:", len(queries))

Разделов: 181 | Запросов: 66


In [71]:
'''
1.0.1 число разделов, категории, форматы и длины текстов
'''
categories = set([i["category"] for i in data["docs"]])
print('>>> Categories:', len(categories))

formats = set([i["format"] for i in data["docs"]])
print('>>> Formats:', len(formats), formats)

>>> Categories: 5
>>> Formats: 3 {'html', 'markdown', 'text'}


In [72]:
'''
1.0.2 Category Stat
'''
from collections import Counter

vals = Counter(item["category"] for item in docs)
print(dict(vals))

vals = Counter(item["format"] for item in docs)
print(dict(vals))

{'Подписка и оплата': 38, 'Функции и контент': 34, 'Аккаунт и профили': 35, 'Устройства и качество': 42, 'Ошибки воспроизведения': 32}
{'text': 132, 'markdown': 28, 'html': 21}


In [73]:
'''
1.1 count junk_ratio - pre
'''
import re

def junk_count(text):
  if not text:
      return 0.0
  # Мусор: всё, что не буква, не цифра, не пробел и не базовый знак препинания
  junk = re.findall(r"[^\w\s.,!?;:()«»—-]", text)
  return len(junk) 


text_len = 0
junk_len = 0
for t in docs:
  text_len += len(t['raw'])
  junk_len += junk_count(str(t['raw']))

print('Junk ratio: ', round(junk_len/text_len, 2), ' | Junk length: ', junk_len,' | Text length: ', text_len)

Junk ratio:  0.05  | Junk length:  7079  | Text length:  145549


In [74]:
'''
1.2 filter HTML
'''
from bs4 import BeautifulSoup  
import trafilatura

def normalize_html(raw):
  soup = BeautifulSoup(raw, "html.parser")
  for tag in soup(["script", "style", "nav", "footer"]):
      tag.decompose()
  for node in soup.select(".cookie"):
      node.decompose()
  return soup.get_text(" ")

for t in docs:
  if t['format'] == 'html':
    txt = trafilatura.extract(t['raw'], favor_precision=False, include_comments=False) 
    if not txt:
    #  print(t['doc_id'])
     txt = normalize_html(t['raw'])
    t['raw'] = txt

# filtered = [i for i in docs if i['raw']]

# write to file
with open("tmp/docs_html.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=4)

/Users/german/work/yai_02/.venv/lib/python3.13/site-packages/urllib3/contrib/socks.py:50: DependencyWarning: SOCKS support in urllib3 requires the installation of optional dependencies: specifically, PySocks.  For more information, see https://urllib3.readthedocs.io/en/latest/advanced-usage.html#socks-proxies
  warnings.warn(


In [75]:
'''
1.3 filter Markdown
'''
import re

for t in docs:
  if t['format'] == 'markdown':
    t['raw'] = re.sub(r"^#.*\n\n", "", t['raw'], flags=re.MULTILINE)
    t['raw'] = re.sub(r"^#.*\n", "", t['raw'], flags=re.MULTILINE)
    t['raw'] = re.sub(r"^#{1,6}\s+", "", t['raw'], flags=re.MULTILINE)

# write to file
with open("tmp/docs_md.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=4)

In [76]:
'''
1.4 encoding, normalize text
'''
import re
import unicodedata
import ftfy

def encode_text(text):
  # Если текст побит — восстанавливаем «кракозябры»
  return ftfy.fix_text(text)

def normalize_text(text):
  # Приводим юникод к единой форме NFC
  text = unicodedata.normalize("NFC", text)

  # Склеиваем слово, разорванное переносом строки
  text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

  # Неразрывные и нулевой ширины пробелы -> обычный пробел
  text = text.replace("\N{NO-BREAK SPACE}", " ")
  text = text.replace("\N{ZERO WIDTH SPACE}", " ")

  # Схлопываем подряд идущие пробелы и переносы в один пробел
  text = re.sub(r"\s+", " ", text)

  return text.strip()


for t in docs:
  if not t['raw']:
    print('>>>', t)
    continue
  if t['format'] != 'html':
    t['raw'] = encode_text(t['raw'])

In [77]:
'''
1.5 remove useless data and put period in the end of item
'''
MIN_LENGTH = 30

filtered = []
for i in docs:
  if len(i['raw']) < MIN_LENGTH \
  or 'Внимание: статья устарела' in i['raw'] \
  or 'выгружено автоматически' in i['raw'] \
  or ' — не публиковать' in i['raw'] \
  or 'Раздел временно недоступен' in i['raw'] \
  or 'Страница не найдена' in i['raw'] \
  or 'Раздел в разработке' in i['raw'] \
  or 'Не pедактировать вpучную' in i['raw'] \
  or 'ORDER BY' in i['raw'] \
  or 'Черновик.' in i['raw'] \
  or 'черновик, не публиковать' in i['raw']:
    continue
  
  i['raw'] = i['raw'].replace("Статья была полезна? Да Нет", "").strip()
  # i['raw'] = i['raw'].replace("— Справка «Фильмоскопа» Главная → Справка →", "").strip()
  # i['raw'] = i['raw'].replace("Мы используем cookie, чтобы сервис работал лучше", "").strip()
  i['raw'] = i['raw'].replace("→", "-").strip()

  if not i['raw'].endswith("."):
    i['raw'] = f"{i['raw']}." 

  filtered.append(i)

for t in filtered:
  t['raw'] = normalize_text(t['raw'])

with open("tmp/docs_filtered.json", "w", encoding="utf-8") as f:
    json.dump(filtered, f, ensure_ascii=False, indent=4)

In [78]:
'''
1.6 Deduplication
'''
import difflib
import itertools

# Recommended Default:
# Catches minor typos, plural forms, or single-word tweaks while keeping distinct sentences separate
threshold=0.85

relevant_docs = [i['relevant_docs'] for i in queries]
relevant_docs = sorted(list(set((list(itertools.chain.from_iterable(relevant_docs))))))

unique = [i['raw'] for i in filtered if i['doc_id'] in relevant_docs]
final = [i for i in filtered if i['doc_id'] in relevant_docs]
items = [i for i in filtered if i not in final]

duplicates = []
for item in items:
  is_duplicate = any(
    difflib.SequenceMatcher(None, item['raw'], existing).ratio() >= threshold
    for existing in unique
  )
  if is_duplicate:
    duplicates.append(item)
    print('>>> duplicate: ', item)
  else:
    unique.append(item['raw'])
    final.append(item)

final_sorted = sorted(final, key=lambda x: int(x["doc_id"]))

with open("1_corpus.json", "w", encoding="utf-8") as f:
    json.dump(final_sorted, f, ensure_ascii=False, indent=4)

>>> duplicate:  {'doc_id': 77, 'title': 'Как смотреть с телефона на большом экране', 'category': 'Устройства и качество', 'format': 'text', 'raw': 'Видео из приложения на телефоне можно отправить на большой экран. Через Chromecast: телефон и телевизор должны быть в одной Wi-Fi сети; нажмите значок трансляции в плеере и выберите устройство - воспроизведение продолжится на телевизоре, а телефон станет пультом. Через AirPlay: на iPhone и iPad нажмите значок AirPlay в плеере и выберите Apple TV или телевизор с поддержкой AirPlay 2. Остановить трансляцию можно тем же значком.'}
>>> duplicate:  {'doc_id': 86, 'title': 'Поддерживаемые устройства', 'category': 'Устройства и качество', 'format': 'markdown', 'raw': 'На каких устройствах работает "Фильмоскоп" Телевизоры Smart TV: Samsung (Tizen, модели с 2018 года), LG (webOS 4.0 и новее), телевизоры и приставки на Android TV 9 и новее. ТВ-приставки: Android TV, Apple TV (tvOS 15 и новее), приставки операторов-партнеров. Телефоны и планшеты: iOS 

In [79]:
'''
1.6.1 Lost relevant_docs
'''
import itertools

doc_ids = [i['doc_id'] for i in final]

relevant_docs = [i['relevant_docs'] for i in queries]
relevant_docs = list(itertools.chain.from_iterable(relevant_docs))

lost = [i for i in relevant_docs if i not in doc_ids]

print(lost)

[]


In [80]:
'''
1.6 Check junk_ratio
'''
import re

def junk_count(text):
  if not text:
      return 0.0
  # Мусор: всё, что не буква, не цифра, не пробел и не базовый знак препинания
  junk = re.findall(r"[^\w\s.,!?;:()«»—-]", text)

  # print('>>> junk', junk)
  return len(junk) 


text_len = 0
junk_len = 0
for t in final:
  text_len += len(t['raw'])
  junk_len += junk_count(str(t['raw']))

print('Junk ratio: ', round(junk_len/text_len, 2), 
      ' | Junk length: ', junk_len, 
      ' | Text length: ', text_len,
      ' | Разделов: ', len(final),
      )

Junk ratio:  0.0  | Junk length:  42  | Text length:  110439  | Разделов:  139


In [81]:

'''
1.7 Corpus Statistics
'''
MIN_LENGTH = 30

texts = [item["raw"] for item in final]

total = 0
total_length = 0
count = 0
for item in texts:
  total += 1
  total_length += len(item)
  if len(item) < MIN_LENGTH:
    count += 1
    print(item)

shortest = min(texts, key=len)
longest = max(texts, key=len)

print(
  '>>> suspicious lengths: ', count, 
  ' | Mean length: ', round(total_length/total, 2), 
  ' | min: ', len(shortest), 
  ' | longest: ', len(longest),
  ' | total docs: ', len(texts),
)
# print('>>> longest:', longest)

>>> suspicious lengths:  0  | Mean length:  794.53  | min:  70  | longest:  1830  | total docs:  139


# Этап 1. Аудит и подготовка корпуса - Report

## Исходный датасет
Разделов: 181 
Запросов: 66 
Text length:  145549

Categories: 5
Docs by Category:
'Подписка и оплата':      38
'Функции и контент':      34 
'Аккаунт и профили':      35 
'Устройства и качество':  42 
'Ошибки воспроизведения': 32

Форматы:
            docs
'text':     132, 
'markdown': 28, 
'html':     21

Junk:
- Junk ratio:  0.05  
- Junk length:  7079  


## Transformations
- HTML -> TEXT = trafilatura + BeautifulSoup
- MD -> TEXT
- Encoding unification
- Text normalization

- Deduplication
threshold=0.85
Recommended Default: Catches minor typos, plural forms, or single-word tweaks while keeping distinct sentences separate

- Suspicious lengths
MIN_LENGTH = 30 - manual check for useful texts

- Manual audit


## Final result
- Suspicious lengths:  0  
- Mean length:  795
- min:  70  / longest:  1830
- total docs:  139
- Junk ratio:  0.0  | Junk length:  42
- No duplicates

## Examples
### text
- Initial:
{
  "doc_id": 43,
  "title": "Аудиодорожки и оригинальная озвучка",
  "category": "Функции и контент",
  "format": "text",
  "raw": "Ð£ Ð¼Ð½Ð¾Ð³Ð¸Ñ ÑÐ¸Ð»ÑÐ¼Ð¾Ð² Ð¸ ÑÐµÑÐ¸Ð°Ð»Ð¾Ð² ÐµÑÑÑ Ð½ÐµÑÐºÐ¾Ð»ÑÐºÐ¾ Ð°ÑÐ´Ð¸Ð¾Ð´Ð¾ÑÐ¾Ð¶ÐµÐº: ÑÑÑÑÐºÐ°Ñ Ð¾Ð·Ð²ÑÑÐºÐ°, Ð¾ÑÐ¸Ð³Ð¸Ð½Ð°Ð»ÑÐ½Ð°Ñ Ð´Ð¾ÑÐ¾Ð¶ÐºÐ°, Ð¸Ð½Ð¾Ð³Ð´Ð° â Ð´Ð¾ÑÐ¾Ð¶ÐºÐ¸ Ð½Ð° Ð´ÑÑÐ³Ð¸Ñ ÑÐ·ÑÐºÐ°Ñ.\n\nÐÐµÑÐµÐºÐ»ÑÑÐ¸ÑÑ Ð´Ð¾ÑÐ¾Ð¶ÐºÑ Ð¼Ð¾Ð¶Ð½Ð¾ Ð¿ÑÑÐ¼Ð¾ Ð² Ð¿Ð»ÐµÐµÑÐµ: Ð¾ÑÐºÑÐ¾Ð¹ÑÐµ Ð¼ÐµÐ½Ñ Â«ÐÑÐ´Ð¸Ð¾Â» Ð¸ Ð²ÑÐ±ÐµÑÐ¸ÑÐµ Ð½ÑÐ¶Ð½ÑÑ. ÐÑÐ±Ð¾Ñ Ð·Ð°Ð¿Ð¾Ð¼Ð¸Ð½Ð°ÐµÑÑÑ Ð´Ð»Ñ ÐºÐ¾Ð½ÐºÑÐµÑÐ½Ð¾Ð³Ð¾ ÑÐ¸Ð»ÑÐ¼Ð°. ÐÐ°Ð±Ð¾Ñ Ð´Ð¾ÑÐ¾Ð¶ÐµÐº Ð·Ð°Ð²Ð¸ÑÐ¸Ñ Ð¾Ñ Ð¿ÑÐ°Ð² Ð½Ð° ÐºÐ¾Ð½ÑÐµÐ½Ñ, Ð¿Ð¾ÑÑÐ¾Ð¼Ñ Ð¾ÑÐ¸Ð³Ð¸Ð½Ð°Ð»ÑÐ½Ð°Ñ Ð¾Ð·Ð²ÑÑÐºÐ° Ð´Ð¾ÑÑÑÐ¿Ð½Ð° Ð½Ðµ Ð²ÐµÐ·Ð´Ðµ: Ð½Ð°Ð»Ð¸ÑÐ¸Ðµ Ð´Ð¾ÑÐ¾Ð¶ÐµÐº ÑÐºÐ°Ð·Ð°Ð½Ð¾ Ð² ÐºÐ°ÑÑÐ¾ÑÐºÐµ ÑÐ¸Ð»ÑÐ¼Ð°. ÐÐ°ÑÑÑÐ¾Ð¹ÐºÐ° Â«ÐÑÐµÐ´Ð¿Ð¾ÑÐ¸ÑÐ°ÐµÐ¼ÑÐ¹ ÑÐ·ÑÐº Ð°ÑÐ´Ð¸Ð¾Â» Ð² Ð¿ÑÐ¾ÑÐ¸Ð»Ðµ Ð·Ð°Ð´Ð°ÑÑ Ð´Ð¾ÑÐ¾Ð¶ÐºÑ Ð¿Ð¾ ÑÐ¼Ð¾Ð»ÑÐ°Ð½Ð¸Ñ."
}
- Final:
{
  "doc_id": 43,
  "title": "Аудиодорожки и оригинальная озвучка",
  "category": "Функции и контент",
  "format": "text",
  "raw": "У многих фильмов и сериалов есть несколько аудиодорожек: русская озвучка, оригинальная дорожка, иногда — дорожки на других языках. Переключить дорожку можно прямо в плеере: откройте меню «Аудио» и выберите нужную. Выбор запоминается для конкретного фильма. Набор дорожек зависит от прав на контент, поэтому оригинальная озвучка доступна не везде: наличие дорожек указано в карточке фильма. Настройка «Предпочитаемый язык аудио» в профиле задаёт дорожку по умолчанию."
}
### html
- Initial:
{
  "doc_id": 57,
  "title": "Политика обработки персональных данных (выдержка)",
  "category": "Аккаунт и профили",
  "format": "html",
  "raw": "<div class=\"kb-article\"><h1>Политика обработки персональных данных (выдержка)</h1>\n<p>Политика обработки персональных данных (выдержка) Сервис обрабатывает данные пользователей для предоставления доступа к контенту, улучшения рекомендаций и обеспечения безопасности аккаунтов. Данные о просмотрах используются в обезличенном виде для аналитики. Пользователь может запросить сведения об обрабатываемых данных через форму обратной связи. Полный текст политики — на странице filmoskop.ru/privacy.</p>\n</div>"
}
- Final:
{
  "doc_id": 57,
  "title": "Политика обработки персональных данных (выдержка)",
  "category": "Аккаунт и профили",
  "format": "html",
  "raw": "Политика обработки персональных данных (выдержка) Сервис обрабатывает данные пользователей для предоставления доступа к контенту, улучшения рекомендаций и обеспечения безопасности аккаунтов. Данные о просмотрах используются в обезличенном виде для аналитики. Пользователь может запросить сведения об обрабатываемых данных через форму обратной связи. Полный текст политики — на странице filmoskop.ru/privacy."
}

### markdown
- Initial:
{
  "doc_id": 98,
  "title": "Универсальный чек-лист при любой ошибке",
  "category": "Ошибки воспроизведения",
  "format": "markdown",
  "raw": "# Универсальный чек-лист при любой ошибке\n\nЕсли на экране появилась ошибка, пройдите базовые шаги — они решают большинство разовых сбоев:\n\n1. Перезапустите фильм, затем приложение. 2. Перезагрузите устройство. 3. Проверьте, что приложение обновлено до свежей версии. 4. Проверьте, что интернет работает на других сайтах и приложениях. 5. Загляните на страницу статуса сервиса: возможно, идут технические работы.\n\nЕсли ошибка осталась, найдите в справке статью по её коду — там будут шаги для конкретной ситуации.\n"
}
- Final:
{
  "doc_id": 98,
  "title": "Универсальный чек-лист при любой ошибке",
  "category": "Ошибки воспроизведения",
  "format": "markdown",
  "raw": "Если на экране появилась ошибка, пройдите базовые шаги — они решают большинство разовых сбоев: 1. Перезапустите фильм, затем приложение. 2. Перезагрузите устройство. 3. Проверьте, что приложение обновлено до свежей версии. 4. Проверьте, что интернет работает на других сайтах и приложениях. 5. Загляните на страницу статуса сервиса: возможно, идут технические работы. Если ошибка осталась, найдите в справке статью по её коду — там будут шаги для конкретной ситуации."
}

## Results
Normalized corpus:  1_corpus.json
Normalized queries: 1_queries.json

# Этап 2. Чанкинг и эмбеддинги


## Install
```
pip install tiktoken razdel
pip install sentence_transformers numpy 
```

## Chunking strategies

Средняя длина раздела - 795 символов. Кириллический текст обычно токенизируется по 1,8–2,5 символа на токен.
Поэтому размер чанка в 400 токенов позволит сохранить целиком большую часть документов. 
В то же время, это позволит не размывать вектора для длинных документов.
Также это безопасный размер для моделей с лимитом в 512 токенов.
Сдвиг выбран как 10% от размера чанков.

1. 400 chars with overlap 
  - со сдвигом: chunk_size = 40
  - within text items

2. sentence based 
  - within text items 
  - max_tokens=400

In [82]:
'''
2.0 Load Docs
'''
import json

with open("1_corpus.json", encoding="utf-8") as f:
  docs = json.load(f)

print("Разделов:", len(docs))

Разделов: 139


In [83]:
'''
2.2 Chunking
'''
import tiktoken
from razdel import sentenize  # сегментация русского текста на предложения

ENC = tiktoken.get_encoding("cl100k_base")

def fixed_size_chunks(text, chunk_size=512, overlap=50):
    """Нарезает текст fixed-size-чанкингом: окно из chunk_size токенов со сдвигом chunk_size − overlap."""
    tokens = ENC.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        window = tokens[start:start + chunk_size]
        chunks.append(ENC.decode(window))
        # Сдвигаем окно: на overlap токенов меньше длины окна
        start += chunk_size - overlap
    return chunks


def sentence_chunks(text, max_tokens=512):
    """Нарезает текст sentence-based-чанкингом: набирает целые предложения, пока не упрётся в бюджет max_tokens."""
    sentences = [s.text for s in sentenize(text)]
    chunks = []
    current = []
    current_tokens = 0
    for sent in sentences:
        sent_tokens = len(ENC.encode(sent))
        # Следующее предложение не влезает — закрываем чанк
        if current and current_tokens + sent_tokens > max_tokens:
            chunks.append(" ".join(current))
            current = []
            current_tokens = 0
        current.append(sent)
        current_tokens += sent_tokens
    if current:
        chunks.append(" ".join(current))
    return chunks


def describe(name, chunks):
    """Печатает сводку по списку чанков: количество, средний размер, текст на стыке соседних чанков."""
    sizes = [len(ENC.encode(c)) for c in chunks]
    avg = sum(sizes) / len(sizes)
    end0 = chunks[0][-24:].replace("\n", " ")
    start1 = chunks[1][:24].replace("\n", " ")
    print(f"{name}")
    print(f"  чанков: {len(chunks)}, средний размер: {avg:.0f}")
    print(f"  размеры, токенов: {sizes}")
    print(f"  чанк 1 кончается на: ...{end0!r}")
    print(f"  чанк 2 начинается с: {start1!r}...")
    print()

In [84]:
'''
2.2.1 Chunking - execute
'''
chunks_fixed = []
chunks_sentence = []
for i in docs:
  fixed = fixed_size_chunks(i['raw'], chunk_size=400, overlap=40)
  chunks_fixed.extend(fixed)
  
  sentence = sentence_chunks(i['raw'], max_tokens=400)
  chunks_sentence.extend(sentence)

# docs as a list
texts = [item["raw"] for item in docs]
texts_str = ", ".join(texts)

print(f"Корпус: {len(ENC.encode(texts_str))} токена\n")
describe("fixed-size (400/40)", chunks_fixed)
describe("sentence-based (400)", chunks_sentence)

Корпус: 48026 токена

fixed-size (400/40)
  чанков: 209, средний размер: 242
  размеры, токенов: [181, 400, 123, 177, 400, 50, 400, 142, 400, 110, 184, 202, 165, 177, 183, 197, 195, 355, 185, 216, 143, 400, 350, 160, 400, 377, 17, 293, 400, 126, 189, 400, 55, 400, 126, 400, 141, 150, 196, 400, 122, 159, 195, 400, 104, 400, 50, 400, 123, 222, 400, 123, 149, 319, 400, 72, 257, 400, 175, 254, 146, 177, 400, 72, 150, 226, 400, 176, 218, 156, 400, 74, 400, 284, 400, 365, 5, 400, 120, 400, 212, 400, 82, 400, 381, 21, 400, 144, 208, 170, 400, 124, 400, 207, 140, 282, 143, 176, 273, 400, 92, 159, 338, 194, 204, 207, 400, 367, 7, 266, 385, 25, 382, 22, 400, 83, 159, 280, 400, 120, 400, 113, 400, 400, 87, 186, 222, 400, 400, 55, 181, 400, 100, 393, 33, 157, 181, 212, 400, 54, 135, 200, 102, 400, 292, 303, 400, 70, 400, 131, 212, 137, 184, 168, 271, 305, 400, 92, 400, 340, 399, 39, 207, 400, 375, 15, 26, 213, 221, 174, 362, 2, 215, 240, 400, 284, 400, 90, 203, 400, 340, 400, 124, 400, 290, 177, 2

In [85]:
'''
2.3 Chunking Statistics
'''

import statistics  # Медиана и другая базовая статистика

MIN_TOKENS = 100   # Чанк короче этого порога — подозрительно короткий


def is_truncated(chunk):
    """Возвращает True, если чанк не заканчивается завершающим знаком препинания, — значит, оборван."""
    if chunk.rstrip().endswith((".", "!", "?", "…")):
      return False
    else: 
      # print('>>> Truncated: ', chunk)
      return True


def evaluate(name, chunks):
    """Считает метрики качества разбивки: распределение размеров, число коротких и оборванных чанков."""
    sizes = [len(ENC.encode(c)) for c in chunks]
    short = sum(size < MIN_TOKENS for size in sizes)
    truncated = sum(is_truncated(c) for c in chunks)
    print(f"{name}:")
    print(f"  чанков: {len(chunks)}")
    print(f"  размеры, токенов: {sorted(sizes)}")
    print(f"  min / медиана / max: {min(sizes)} / "
          f"{statistics.median(sizes):.0f} / {max(sizes)}")
    print(f"  короче {MIN_TOKENS} токенов: {short}")
    print(f"  обрезаны посреди предложения: {truncated}")

    # for c in chunks:
    #   if len(ENC.encode(c))  < MIN_TOKENS:
    #     print('>>> small size: ', c)
    #    if is_truncated(c):
    #     print('>>> truncated: ', c)


evaluate("fixed-size (400/40)", chunks_fixed)
print()
evaluate("sentence-based (400)", chunks_sentence)

fixed-size (400/40):
  чанков: 209
  размеры, токенов: [2, 5, 7, 15, 17, 21, 22, 22, 25, 26, 33, 39, 40, 45, 50, 50, 54, 55, 55, 55, 70, 72, 72, 74, 82, 83, 87, 90, 92, 92, 97, 100, 102, 104, 110, 113, 120, 120, 122, 123, 123, 123, 124, 124, 126, 126, 131, 131, 135, 137, 140, 141, 142, 143, 143, 144, 146, 149, 150, 150, 156, 157, 159, 159, 159, 160, 161, 163, 164, 165, 168, 170, 174, 174, 175, 176, 176, 177, 177, 177, 177, 181, 181, 181, 183, 184, 184, 185, 186, 189, 194, 195, 195, 196, 197, 199, 200, 202, 203, 204, 207, 207, 207, 208, 212, 212, 212, 213, 215, 216, 218, 221, 222, 222, 226, 240, 254, 257, 266, 271, 273, 278, 279, 280, 282, 284, 284, 290, 292, 293, 303, 305, 319, 338, 340, 340, 350, 354, 355, 362, 365, 367, 375, 377, 381, 382, 382, 385, 393, 399, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400

In [86]:
'''
2.4 Save chunks to files - sentence_based
'''
chunks_final = []
id = 0
for i in docs:
  chunks = sentence_chunks(i['raw'], max_tokens=400)

  for j in chunks:
    chunk =   {
      "id":       id,
      "doc_id":   i["doc_id"],
      "title":    i['title'],
      "category": i['category'],
      "format":   i['format'],
      'text':     j,
    }
    chunks_final.append(chunk)
    id += 1
    # print(chunk)
    
with open("2_chunks.json", "w", encoding="utf-8") as f:
  json.dump(chunks_final, f, ensure_ascii=False, indent=4)

In [87]:
'''
2.4.1 Save chunks to files - fixed
'''
chunks_fixed = []
id = 0
for i in docs:
  chunks = fixed_size_chunks(i['raw'], chunk_size=400, overlap=40)

  for j in chunks:
    chunk =   {
      "id":       id,
      "doc_id":   i["doc_id"],
      "title":    i['title'],
      "category": i['category'],
      "format":   i['format'],
      'text':     j,
    }
    chunks_fixed.append(chunk)
    id += 1
    # print(chunk)
    
with open("2_chunks_fixed.json", "w", encoding="utf-8") as f:
  json.dump(chunks_fixed, f, ensure_ascii=False, indent=4)

In [88]:
'''
2.5 Load chunks for Embedding - sentence_based
'''
import json

# --- Загружаем корпус: список из 100 словарей с текстом и метаданными ---
with open("2_chunks.json", encoding="utf-8") as f:
  chunks = json.load(f)

texts = [c["text"] for c in chunks]
metas = [{k: c[k] for k in ("id", "doc_id", "category", "format", "title")}
         for c in chunks]

print("Разделов:", len(chunks))

Разделов: 195


In [89]:
'''
2.5.1 Load chunks for Embedding - fixed
'''
import json

# --- Загружаем корпус: список из 100 словарей с текстом и метаданными ---
with open("2_chunks_fixed.json", encoding="utf-8") as f:
  chunks_fixed = json.load(f)

texts_fixed = [c["text"] for c in chunks_fixed]
metas_fixed = [{k: c[k] for k in ("id", "doc_id", "category", "format", "title")}
         for c in chunks_fixed]

print("Разделов:", len(chunks_fixed))

Разделов: 209


In [90]:
'''
2.6 Embed chunks - sentence-based
'''
import numpy as np
from sentence_transformers import SentenceTransformer

def embed_passages(model, texts):
    """Эмбеддинги документов через multilingual-e5: префикс 'passage: '."""
    inputs = ["passage: " + t for t in texts]
    return model.encode(inputs, normalize_embeddings=True, batch_size=64)


# --- Считаем эмбеддинги корпуса (e5-base, 768 координат) и сохраняем ---
embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")
corpus_vecs = np.ascontiguousarray(embed_passages(embedder, texts), dtype="float32")
np.save("2_kb_vecs.npy", corpus_vecs)   # Эмбеддинги — в файл, переиспользуем

print(f"Чанков: {len(texts)}, эмбеддинги: {corpus_vecs.shape}")
print("Пример метаданных:", metas[1])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Чанков: 195, эмбеддинги: (195, 768)
Пример метаданных: {'id': 1, 'doc_id': 1, 'category': 'Функции и контент', 'format': 'text', 'title': 'Синхронизация между устройствами'}


In [91]:
'''
2.6.1 Embed chunks - fixed
'''
import numpy as np

corpus_vecs_fixed = np.ascontiguousarray(embed_passages(embedder, texts_fixed), dtype="float32")
np.save("2_kb_vecs_fixed.npy", corpus_vecs_fixed)   

print(f"Чанков: {len(texts_fixed)}, эмбеддинги: {corpus_vecs_fixed.shape}")
print("Пример метаданных:", metas_fixed[1])

Чанков: 209, эмбеддинги: (209, 768)
Пример метаданных: {'id': 1, 'doc_id': 1, 'category': 'Функции и контент', 'format': 'text', 'title': 'Синхронизация между устройствами'}


# Этап 2. Чанкинг и эмбеддинги - Report


## statistics
### fixed-size (400/40):
-  чанков: 209
-  min / медиана / max: 2 / 212 / 400
-  короче 100 токенов: 31
-  ** обрезаны посреди предложения: 55 **

Example of sentence split:
    {
        "id": 1,
        "doc_id": 1,
        "title": "Синхронизация между устройствами",
        "category": "Функции и контент",
        "format": "text",
        "text": "Всё, что вы делаете в «Фильмоскопе», хранится не на устройстве, а в аккаунте. Поэтому телефон, планшет, браузер и телевизор показывают одно и то же: вошли — и сервис выглядит привычно, даже если устройство чужое. Общее для аккаунта: подписка и тариф, способ оплаты, список привязанных устройств. Это не зависит ни от профиля, ни от того, откуда вы зашли. Общее внутри профиля: список «Моё», оценки «Нравится» и «Не нравится», купленные фильмы и настройки самого профиля — язык интерфейса, автовоспроизведение, аватар. Отметили фильм на телефоне — он окажется в «Моём» и на телевизоре. При этом у каждого профиля всё своё: соседний профиль ваших оценок и вашего списка не увидит. Остаётся на устройстве: скачанные файлы и настройки, которые относятся к самой технике, — громкость, вывод звука, параметры изображения. Переносить их некуда, поэтому на нов"
    },
    {
        "id": 2,
        "doc_id": 1,
        "title": "Синхронизация между устройствами",
        "category": "Функции и контент",
        "format": "text",
        "text": "нике, — громкость, вывод звука, параметры изображения. Переносить их некуда, поэтому на новом устройстве они задаются заново. Синхронизация требует интернета. Если устройство было офлайн, изменения доедут до аккаунта при следующем подключении, а пока экраны на разных устройствах могут расходиться."
    },


### sentence-based (400):
-  чанков: 195
-  min / медиана / max: 26 / 221 / 404
-  короче 100 токенов: 23
-  ** обрезаны посреди предложения: 0 **

## Ручной просмотр чанков выполнен выборочно.


## Стратегия эмбединга
Далее используется sentence-based embedding - ввиду того, что он не разбивает на части предложения.


## Выбор индекса
Будет использоваться HNSW (Hierarchical Navigable Small World) в связке с косинусным сходством (Cosine Similarity) 
- Высокая точность (Recall) - для приближенного поиска 
- Поддержка инкрементальных обновлений
- Для небольщого корпуса памяти должно быть достаточно


## Results
2_chunks.json
2_kb_vecs.npy

# Этап 3. Поисковая инфраструктура и retrieval

## Vector DB
В качестве хранилища выбрано Chroma DB.
- для удобства хранения и поиска по индексу в БД
- для возможности поиска внутри Категорий (и другим метаданным)
- Qdrant - более требовательная к ресурсам технология.

При переходе в промышленную эксплуатацию предполагается перейти на Qdrant - для удобства сопровождения и эксплуатации.


# install
pip install chromadb

In [92]:
'''
3.1 Load vectors to DB - sentence-based
'''
import chromadb
from sentence_transformers import SentenceTransformer

# --- Создаём локальную коллекцию Chroma ---
client = chromadb.PersistentClient(path="chroma_kb")     # Хранение на диске
try:
    client.delete_collection("knowledge_base")           # Чистим прошлый прогон
except Exception:
    pass

col = client.create_collection(
    name="knowledge_base",
    configuration={"hnsw": {"space": "cosine"}})         # Косинусная близость

col.add(
    ids=[str(i) for i in range(len(texts))],             # Уникальные id
    embeddings=corpus_vecs.tolist(),                     # Векторы чанков
    documents=texts,                                     # Сами тексты
    metadatas=metas,                                     # Метаданные
)
print("Записей в коллекции:", col.count())

Записей в коллекции: 195


In [93]:
'''
3.1.1 Load vectors to DB - fixed
'''
try:
    client.delete_collection("knowledge_base_fixed")           # Чистим прошлый прогон
except Exception:
    pass

col_fixed = client.create_collection(
    name="knowledge_base_fixed",
    configuration={"hnsw": {"space": "cosine"}})         # Косинусная близость

col_fixed.add(
    ids=[str(i) for i in range(len(texts_fixed))],             # Уникальные id
    embeddings=corpus_vecs_fixed.tolist(),                     # Векторы чанков
    documents=texts_fixed,                                     # Сами тексты
    metadatas=metas_fixed,                                     # Метаданные
)
print("Записей в коллекции:", col_fixed.count())

Записей в коллекции: 209


In [94]:
'''
3.1.1 Test 1 search
Correct doc: #140
'''
from sentence_transformers import SentenceTransformer

QUERY = "боюсь, что ребёнок случайно купит фильм с моей карты"
e5 = SentenceTransformer("intfloat/multilingual-e5-base")
q_vec = e5.encode(["query: " + QUERY], normalize_embeddings=True)[0]

res = col.query(
  query_embeddings=[q_vec.tolist()],
  n_results=5,
)           

print(f"\nЗапрос: {QUERY}")
for doc, m, dist in zip(res["documents"][0], 
                        res["metadatas"][0],
                        res["distances"][0]):
  print(f"  dist={dist:.3f} [id: {m['id']}] [doc_id: {m['doc_id']}] [title: {m['title']}] {doc}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Запрос: боюсь, что ребёнок случайно купит фильм с моей карты
  dist=0.162 [id: 112] [doc_id: 108] [title: Родительский контроль: обзор] Родительский контроль — общее название для настроек, которые ограничивают то, что ребёнок видит и делает в онлайн-кинотеатре. Почти везде он складывается из двух частей: фильтра по возрастным рейтингам и ограничения платных действий. Возрастной рейтинг присваивается фильму при выдаче прокатного удостоверения, сервис только показывает его в карточке и использует как фильтр. Рейтинг относится к фильму целиком, поэтому добрый на вид мультфильм с одной тревожной сценой может стоять на ступень выше, чем ожидают родители. Ограничение платных действий закрывает вторую типичную неприятность — оплату случайным нажатием. На телевизоре это особенно легко: карточка новинки и кнопка с ценой находятся в двух движениях пульта. Сервисы решают это по-разному: одни убирают раздел с покупками из детского режима, другие просят подтверждение перед оплатой.
  dist=0.169 [i

In [95]:
'''
3.1.2 Example search with metadata
Correct doc: #146
'''
from sentence_transformers import SentenceTransformer

QUERY = "боюсь, что ребёнок случайно купит фильм с моей карты"
e5 = SentenceTransformer("intfloat/multilingual-e5-base")
q_vec = e5.encode(["query: " + QUERY], normalize_embeddings=True)[0]

res = col.query(
  query_embeddings=[q_vec.tolist()],
  n_results=3,
  where={'category': 'Устройства и качество'}
)           

print(f"\nЗапрос: {QUERY}")
for doc, m, dist in zip(res["documents"][0], 
                        res["metadatas"][0],
                        res["distances"][0]):
  print(f"  dist={dist:.3f} [id: {m['id']}] [doc_id: {m['doc_id']}] [title: {m['title']}] {doc}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Запрос: боюсь, что ребёнок случайно купит фильм с моей карты
  dist=0.190 [id: 22] [doc_id: 20] [title: Просмотр за границей] После возвращения домой каталог автоматически станет прежним. Страна определяется по подключению: сервис ориентируется на сеть, из которой вы вышли в интернет. Поэтому каталог меняется после подключения к местному Wi-Fi или мобильной сети, а не в момент пересечения границы. Если нужного фильма в каталоге страны пребывания нет, добавьте его в список «Моё» — карточка сохранится, и после возвращения фильм снова станет доступен. Заново искать его в каталоге не придётся. То же касается сериалов: отмеченные позиции никуда не пропадают из списка. Когда после возвращения домой каталог остался прежним, перезапустите приложение и проверьте, что подключение идёт через домашнюю сеть. Если это не помогает, напишите в поддержку и укажите страну, из которой заходите.
  dist=0.192 [id: 43] [doc_id: 40] [title: Защита контента: зачем она нужна] Если фильм не запускается и плеер

In [96]:
'''
3.2 Retrieve - dict
'''
from sentence_transformers import SentenceTransformer

# preliminary selection
THRESHOLD = 0.83
K = 5

embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="cpu")

def retrieve_dict(query, col=col, embedder=embedder, k=K, threshold=THRESHOLD):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append({
        'id': m['id'],
        'doc_id': m['doc_id'], 
        'similarity': round(sim, 4), 
        'text': doc, 
      }
    )
  return out

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [97]:
'''
3.2.1 Test: 3 queries 
'''
import json

# load queries
with open("1_queries.json", encoding="utf-8") as f:
  data = json.load(f)

# filter GAPs
queries = [u for u in data if u.get("kind") != "gap"]


for qi, q in enumerate(queries[:3]):
  ret = retrieve_dict(q["query"])
  print(f"\n[{qi}] {q['query']} {q['relevant_docs']}")
  if not ret:
    print("    (ничего выше порога — пайплайн вернул пусто)")
  for i in ret:
    print(f"    sim={i['similarity']:.3f} #{i['doc_id']} {i['text'][:58]} #{i['id']}")


[0] что означает ошибка P-102 [85]
    sim=0.870 #85 Ошибка P-102 появляется, когда число одновременных просмот #92
    sim=0.861 #178 Ошибка P-109 появляется, когда на устройстве не хватает св #191
    sim=0.858 #137 Ошибка P-106 появляется, когда плееру не удаётся загрузить #146
    sim=0.856 #138 Ошибка P-104 появляется, когда устройство не поддерживает  #147
    sim=0.856 #129 Ошибка P-105 появляется, когда на устройстве установлена у #137

[1] какая версия webOS или Tizen нужна для приложения на телевизоре [146]
    sim=0.878 #146 Телевизоры Smart TV: Samsung (Tizen, модели с 2018 года),  #157
    sim=0.838 #129 Ошибка P-105 появляется, когда на устройстве установлена у #137
    sim=0.836 #172 Приложение на телевизоре обновляется само, если в магазине #183
    sim=0.836 #139 Если вы не знаете, к какой платформе относится ваш телевиз #148
    sim=0.834 #176 Если приложение на телевизоре закрывается само или зависае #189

[2] обязателен ли HDCP 2.2 для просмотра в 4K [110]
    sim=

In [98]:
'''
3.3 Test: compare with relevant docs
'''
relevant = {qi: set(q["relevant_docs"]) for qi, q in enumerate(queries)}

problem = []
for qi, q in enumerate(queries):
    ret = retrieve_dict(q["query"])
    # hits = retrieve(q["query"], k=5, threshold=0.0)
    top_ids = [i['doc_id'] for i in ret]
    rel = relevant[qi]
    is_problem = (not top_ids) or (top_ids[0] not in rel)
    if is_problem:
        problem.append(qi)

print("Проблемные запросы:", problem)

for qi in problem:
    q = queries[qi]
    rel = relevant[qi]
    ret = retrieve_dict(q["query"], k=5, threshold=0.75)
    print(f"\n[{qi}] {q['query']}  (релевантные id: {sorted(rel) or 'нет'})")
    for i in ret:
        mark = "+" if i['doc_id'] in rel else " "
        print(f"    {mark} sim={i['similarity']:.3f} #{i['id']} {i['text'][:60]}")

Проблемные запросы: [3, 10, 13, 15, 18, 19, 20, 22, 24, 25, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 41, 43, 44, 46, 47, 48, 50, 51, 53, 54]

[3] как подключить саундбар по HDMI ARC  (релевантные id: [22])
      sim=0.842 #133 Если при воспроизведении звук идет, а изображения нет — экра
      sim=0.834 #77 Объёмный звук дома собирают двумя способами, и выбор между н
    + sim=0.816 #24 Обозначение 5.1 описывает раскладку каналов: пять основных к
      sim=0.812 #56 Встроенные динамики телевизора делают тонкими вместе с корпу
      sim=0.810 #160 У вас уже есть всё для домашнего кинотеатра — просто вы назы

[10] сколько профилей можно создать в одном аккаунте  (релевантные id: [63])
      sim=0.893 #70 Сколько профилей помещается в один аккаунт и как их создать,
    + sim=0.874 #71 Профиль — это отдельное пространство внутри одного аккаунта:
      sim=0.868 #69 Аккаунт и профиль — не одно и то же. Аккаунт заводится на но
    + sim=0.854 #72 Если человек пользуется сервисом изредк

In [99]:
'''
3.4 recall@5, precision@5, MRR
'''
def recall_at_k(found, relevant, k=5):
  """Доля релевантных документов в первых k результатах."""
  hits = set(found[:k]) & relevant
  return len(hits) / len(relevant) if relevant else 0.0

def precision_at_k(found_ids, relevant_ids, k=5):
  """Доля релевантных среди k выданных чанков."""
  hits = [i for i in found_ids[:k] if i in relevant_ids]
  return len(hits) / k

def reciprocal_rank(found_ids, relevant_ids):
    """Обратный ранг первого релевантного чанка; 0, если его нет в выдаче."""
    for rank, i in enumerate(found_ids, start=1):
        if i in relevant_ids:
            return 1.0 / rank
    return 0.0

In [100]:
'''
3.4.1 check recall@5, precision@5, MRR
'''
import numpy as np

def basic_search(query, k=5, threshold=0.8, col=col):
  ret = retrieve_dict(query, col=col, k=k, threshold=threshold)
  doc_ids = [i['doc_id'] for i in ret]
  return doc_ids


K = 5
rows = []
for q in queries:
    rel = set(q["relevant_docs"])
    found = basic_search(q["query"])
    rows.append({
        "n_rel": len(rel),
        "p": precision_at_k(found, rel, K),
        "r": recall_at_k(found, rel, K),
        "rr": reciprocal_rank(found, rel),
    })

print('>>> Basic Search')
print(f"{'запрос':<44}{'|R|':>4}{'P@5':>7}{'R@5':>7}{'RR':>7}")
for row, q in zip(rows, queries):
    print(f"{q['query'][:42]:<44}{row['n_rel']:>4}"
          f"{row['p']:>7.2f}{row['r']:>7.2f}{row['rr']:>7.2f}")

print(f"\nprecision@5 = {np.mean([r['p'] for r in rows]):.3f}")
print(f"recall@5    = {np.mean([r['r'] for r in rows]):.3f}")
print(f"MRR         = {np.mean([r['rr'] for r in rows]):.3f}")

>>> Basic Search
запрос                                       |R|    P@5    R@5     RR
что означает ошибка P-102                      1   0.20   1.00   1.00
какая версия webOS или Tizen нужна для при     1   0.20   1.00   1.00
обязателен ли HDCP 2.2 для просмотра в 4K      1   0.40   1.00   1.00
как подключить саундбар по HDMI ARC            1   0.20   1.00   0.33
что означает ошибка P-109                      1   0.20   1.00   1.00
что значит ошибка P-111                        1   0.20   1.00   1.00
сколько стоит тариф Максимум и что в него      1   0.40   1.00   1.00
сколько дней длится пробный период             1   0.40   1.00   1.00
какая скорость интернета нужна для 4K          1   0.20   1.00   1.00
есть ли приложение Фильмоскопа для Nintend     1   0.20   1.00   1.00
сколько профилей можно создать в одном акк     1   0.40   1.00   0.50
на каком тарифе доступен звук 5.1              2   0.40   0.50   1.00
сколько устройств можно привязать к аккаун     1   0.20   1.00   1.00
ско

In [101]:
'''
3.4.1 check recall@5, precision@5, MRR - fixed
'''
import numpy as np

K = 5
rows = []
for q in queries:
    rel = set(q["relevant_docs"])
    found = basic_search(q["query"], col=col_fixed)
    rows.append({
        "n_rel": len(rel),
        "p": precision_at_k(found, rel, K),
        "r": recall_at_k(found, rel, K),
        "rr": reciprocal_rank(found, rel),
    })

print('>>> Basic Search')
print(f"{'запрос':<44}{'|R|':>4}{'P@5':>7}{'R@5':>7}{'RR':>7}")
for row, q in zip(rows, queries):
    print(f"{q['query'][:42]:<44}{row['n_rel']:>4}"
          f"{row['p']:>7.2f}{row['r']:>7.2f}{row['rr']:>7.2f}")

print(f"\nprecision@5 = {np.mean([r['p'] for r in rows]):.3f}")
print(f"recall@5    = {np.mean([r['r'] for r in rows]):.3f}")
print(f"MRR         = {np.mean([r['rr'] for r in rows]):.3f}")

>>> Basic Search
запрос                                       |R|    P@5    R@5     RR
что означает ошибка P-102                      1   0.20   1.00   1.00
какая версия webOS или Tizen нужна для при     1   0.20   1.00   1.00
обязателен ли HDCP 2.2 для просмотра в 4K      1   0.40   1.00   1.00
как подключить саундбар по HDMI ARC            1   0.40   1.00   0.33
что означает ошибка P-109                      1   0.20   1.00   1.00
что значит ошибка P-111                        1   0.20   1.00   1.00
сколько стоит тариф Максимум и что в него      1   0.40   1.00   0.50
сколько дней длится пробный период             1   0.40   1.00   1.00
какая скорость интернета нужна для 4K          1   0.20   1.00   1.00
есть ли приложение Фильмоскопа для Nintend     1   0.20   1.00   1.00
сколько профилей можно создать в одном акк     1   0.40   1.00   0.33
на каком тарифе доступен звук 5.1              2   0.40   0.50   1.00
сколько устройств можно привязать к аккаун     1   0.20   1.00   1.00
ско

In [102]:
'''
3.5 analize K & threshhold
'''
for tr in [0.7, 0.75, 0.8, 0.83]:
  for K in [1, 3, 5, 10, 20]:
    rows = []
    for q in queries:
      rel = set(q["relevant_docs"])
      found = basic_search(q["query"], k=K, threshold=tr)
      rows.append(recall_at_k(found, rel, K))

    print(f"threshold = {tr} | K = {K} | recall@{K} = {np.mean(rows):.3f}")

threshold = 0.7 | K = 1 | recall@1 = 0.384
threshold = 0.7 | K = 3 | recall@3 = 0.679
threshold = 0.7 | K = 5 | recall@5 = 0.768
threshold = 0.7 | K = 10 | recall@10 = 0.911
threshold = 0.7 | K = 20 | recall@20 = 0.973
threshold = 0.75 | K = 1 | recall@1 = 0.384
threshold = 0.75 | K = 3 | recall@3 = 0.679
threshold = 0.75 | K = 5 | recall@5 = 0.768
threshold = 0.75 | K = 10 | recall@10 = 0.911
threshold = 0.75 | K = 20 | recall@20 = 0.973
threshold = 0.8 | K = 1 | recall@1 = 0.384
threshold = 0.8 | K = 3 | recall@3 = 0.661
threshold = 0.8 | K = 5 | recall@5 = 0.732
threshold = 0.8 | K = 10 | recall@10 = 0.875
threshold = 0.8 | K = 20 | recall@20 = 0.938
threshold = 0.83 | K = 1 | recall@1 = 0.366
threshold = 0.83 | K = 3 | recall@3 = 0.580
threshold = 0.83 | K = 5 | recall@5 = 0.598
threshold = 0.83 | K = 10 | recall@10 = 0.670
threshold = 0.83 | K = 20 | recall@20 = 0.696


In [103]:
'''
3.4.1 check recall@5, precision@5, MRR 
  threshold=0.75
  sentence-based
'''
import numpy as np

K = 5
rows = []
for q in queries:
    rel = set(q["relevant_docs"])
    found = basic_search(q["query"], threshold=0.75, col=col)
    rows.append({
        "n_rel": len(rel),
        "p": precision_at_k(found, rel, K),
        "r": recall_at_k(found, rel, K),
        "rr": reciprocal_rank(found, rel),
    })

print('>>> Basic Search')
print(f"{'запрос':<44}{'|R|':>4}{'P@5':>7}{'R@5':>7}{'RR':>7}")
for row, q in zip(rows, queries):
    print(f"{q['query'][:42]:<44}{row['n_rel']:>4}"
          f"{row['p']:>7.2f}{row['r']:>7.2f}{row['rr']:>7.2f}")

print(f"\nprecision@5 = {np.mean([r['p'] for r in rows]):.3f}")
print(f"recall@5    = {np.mean([r['r'] for r in rows]):.3f}")
print(f"MRR         = {np.mean([r['rr'] for r in rows]):.3f}")

>>> Basic Search
запрос                                       |R|    P@5    R@5     RR
что означает ошибка P-102                      1   0.20   1.00   1.00
какая версия webOS или Tizen нужна для при     1   0.20   1.00   1.00
обязателен ли HDCP 2.2 для просмотра в 4K      1   0.40   1.00   1.00
как подключить саундбар по HDMI ARC            1   0.20   1.00   0.33
что означает ошибка P-109                      1   0.20   1.00   1.00
что значит ошибка P-111                        1   0.20   1.00   1.00
сколько стоит тариф Максимум и что в него      1   0.40   1.00   1.00
сколько дней длится пробный период             1   0.40   1.00   1.00
какая скорость интернета нужна для 4K          1   0.20   1.00   1.00
есть ли приложение Фильмоскопа для Nintend     1   0.20   1.00   1.00
сколько профилей можно создать в одном акк     1   0.40   1.00   0.50
на каком тарифе доступен звук 5.1              2   0.40   0.50   1.00
сколько устройств можно привязать к аккаун     1   0.20   1.00   1.00
ско

# Этап 3. Поисковая инфраструктура и retrieval - Report

Требования: recall@5 >= 0,85

## chunking strategy
```
(threshold = 0.83)
- fixed size
precision@5 = 0.243
recall@5    = 0.732
MRR         = 0.590

- sentence-based
precision@5 = 0.250
recall@5    = 0.732
MRR         = 0.615

** Метрики для sentence-based и fixed size chunking почти совпадают **

## Threshold
threshold = 0.7   recall@5 = 0.768
threshold = 0.75  recall@5 = 0.768
threshold = 0.8   recall@5 = 0.732
threshold = 0.83  recall@5 = 0.598

На основе анализа выбраны параметры:
Threshold = 0.75 - max, чтобы не упустить правильные варианты

## Final metrics to optimize
precision@5 = 0.257
recall@5    = 0.768
MRR         = 0.628
```

## Example search with metadata
3.1.2 

## Пересмотр параметров
Chroma DB - до 500 тыс. чанков, или до выхода в пром.эксплуатацию.

Размер вектора float32: 3 КБ (768 * 4 байта). 
500 тыс. чанков: ~1.65 ГБ RAM - предположительно также предел для MacBook M4 16gb.

# Этап 4. Диагноз и улучшение retrieval

## Optimixation strategy
Based on the data:
- recall@5 = 0.723 vs. recall@20 = 0.929
** cross-encoder reranking ** 
is selected for retrieval optimization

Threshhold = 0.75 

In [104]:
''' 4.0 Vector DB '''
import chromadb
client_db = chromadb.PersistentClient(path="chroma_kb")
col = client_db.get_collection(name="knowledge_base")  

In [105]:
'''
4.1.0 Retrieve
'''
from sentence_transformers import SentenceTransformer

THRESHOLD = 0.75
K = 5

embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")

def retrieve_dict(query, col=col, embedder=embedder, k=K, threshold=THRESHOLD):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append({
        'id': m['id'],
        'doc_id': m['doc_id'], 
        'similarity': round(sim, 4), 
        'text': doc, 
      }
    )
  return out

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [106]:
'''
4.1 Reranking func
'''
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="mps")

def retrieve_rerank(query, k=5):
  ret = retrieve_dict(query, k=20)
  cands = [i['text'] for i in ret]
  scores = reranker.predict([(query, c) for c in cands])
  for i,s in enumerate(scores, 0):
    ret[i]['score'] = s
  sorted_ret = sorted(ret, key=lambda x: x["score"], reverse=True)
  return sorted_ret[:k]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [107]:
'''
4.1.1 Reranking Statistics
'''
import numpy as np

with open("1_corpus.json", encoding="utf-8") as f:
  docs = json.load(f)
corpus = [i['raw'] for i in docs]

with open("1_queries.json", encoding="utf-8") as f:
  queries = json.load(f)
queries = [i for i in queries if i['kind'] != 'gap']
# filter GAPs
queries_txt = [i['query'] for i in queries if i['kind'] != 'gap']

rows = []
for q in queries:
  rel = set(q["relevant_docs"])
  found = retrieve_rerank(q["query"])
  ids = [i['doc_id'] for i in found]
  rows.append({
      "n_rel": len(rel),
      "p": precision_at_k(ids, rel, K),
      "r": recall_at_k(ids, rel, K),
      "rr": reciprocal_rank(ids, rel),
  })

print('>>> Reranking')
print(f"{'запрос':<44}{'|R|':>4}{'P@5':>7}{'R@5':>7}{'RR':>7}")
for row, q in zip(rows, queries):
    print(f"{q['query'][:42]:<44}{q['kind']}{row['n_rel']:>4}"
          f"{row['p']:>7.2f}{row['r']:>7.2f}{row['rr']:>7.2f}")

print(f"\nprecision@5 = {np.mean([r['p'] for r in rows]):.3f}")
print(f"recall@5    = {np.mean([r['r'] for r in rows]):.3f}")
print(f"MRR         = {np.mean([r['rr'] for r in rows]):.3f}")

>>> Reranking
запрос                                       |R|    P@5    R@5     RR
что означает ошибка P-102                   exact   1   0.20   1.00   1.00
какая версия webOS или Tizen нужна для при  exact   1   0.20   1.00   1.00
обязателен ли HDCP 2.2 для просмотра в 4K   exact   1   0.40   1.00   1.00
как подключить саундбар по HDMI ARC         exact   1   0.20   1.00   1.00
что означает ошибка P-109                   exact   1   0.20   1.00   1.00
что значит ошибка P-111                     exact   1   0.20   1.00   1.00
сколько стоит тариф Максимум и что в него   exact   1   0.40   1.00   1.00
сколько дней длится пробный период          exact   1   0.40   1.00   1.00
какая скорость интернета нужна для 4K       exact   1   0.20   1.00   1.00
есть ли приложение Фильмоскопа для Nintend  exact   1   0.20   1.00   1.00
сколько профилей можно создать в одном акк  exact   1   0.40   1.00   1.00
на каком тарифе доступен звук 5.1           exact   2   0.60   1.00   1.00
сколько устройст

In [108]:
'''
4.1.2 Reranking test - evaluate queries for manual tests
'''
import numpy as np

for qi in range(len(queries)):
    reranked = retrieve_rerank(queries[qi]['query'])
    rel = queries[qi]['relevant_docs']
    print(f"  После реранкинга top-5: {queries[qi]['query']}")
    for rank, i in enumerate(reranked, 1):
        mark = "+" if i['doc_id'] in rel else " "
        print(f"    {mark} {rank}. #{i['doc_id']:03d} {i['text'][:70]}...")


def recall_at_k(found, relevant, k=5):
  """Доля релевантных документов в первых k результатах."""
  hits = set(found[:k]) & relevant
  return len(hits) / len(relevant) if relevant else 0.0

  После реранкинга top-5: что означает ошибка P-102
    + 1. #085 Ошибка P-102 появляется, когда число одновременных просмотров на аккау...
      2. #165 Ошибка P-112 появляется, когда аккаунт заблокирован за нарушение польз...
      3. #090 Ошибка P-110 появляется, когда происходит сбой на стороне сервиса: с в...
      4. #094 Ошибка P-101 появляется, когда приложению не удаётся соединиться с сер...
      5. #102 Если просмотр прерывается, плеер показывает короткий код вида P-1xx. Б...
  После реранкинга top-5: какая версия webOS или Tizen нужна для приложения на телевизоре
    + 1. #146 Телевизоры Smart TV: Samsung (Tizen, модели с 2018 года), LG (webOS 4....
      2. #172 Приложение на телевизоре обновляется само, если в магазине включено ав...
      3. #139 Если вы не знаете, к какой платформе относится ваш телевизор, найдите ...
      4. #040 Фильмы в онлайн-кинотеатрах передаются в зашифрованном виде. Это услов...
      5. #066 Если поиск в магазине телевизора ничего не находит, 

# Этап 4. Диагноз и улучшение retrieval - Report

## Реализован варианты улучшения:
- реранкер cross-encoder

## Results
- initial
precision@5 = 0.257
recall@5    = 0.768
MRR         = 0.628

- final
precision@5 = 0.321
recall@5    = 0.946
MRR         = 0.829

## Initial requirement has been met
recall@5    = 0.946 > 0.85

The price of optimization is latency for reranking (~90 times):
Statistics test for all queries: ~90 sek vs. 1 sek (MacBook M4 16gb)


# Этап 5. Генерация ответов


# install
```
pip install openai
```

In [110]:
'''
5.0 LLM Generate
'''
from openai import OpenAI

client = OpenAI(
  base_url="http://127.0.0.1:9931/v1", 
  # base_url="http://localhost:8080/v1", 
  api_key="not-needed",
)
LLM = "qwen3.5"                                       # Имя сервер игнорирует
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}


def generate(messages, max_tokens=160):
    """Один вызов локальной LLM. temperature=0 — для воспроизводимости,
    режим рассуждения Qwen3.5 отключён, поэтому сразу приходит готовый текст."""
    resp = client.chat.completions.create(
        model=LLM, messages=messages, temperature=0.0,
        max_tokens=max_tokens, extra_body=NO_THINK)
    return resp.choices[0].message.content.strip()

In [111]:
'''
5.0.1 load all queries
'''
import json

with open("1_queries.json", encoding="utf-8") as f:
  queries = json.load(f)

In [112]:
'''
5.1 Pipeline
'''
SYSTEM = (
  "Ты — ассистент поддержки. Отвечай ТОЛЬКО по приведённым фрагментам базы знаний. "
  "Если ответа нет, ответь точно: «В базе знаний ответа нет». "
  "Ничего не добавляй."
)

def format_items(items):
  if len(items) <= 2:
      return '\n'.join(items)
  reordered = [items[0]] + items[2:] + [items[1]]
  return '\n'.join(reordered)

def build_messages(query, chunk_dicts):
  """Собирает промпт: system-инструкция + нумерованный контекст + вопрос."""
  reordered = [chunk_dicts[0]] + chunk_dicts[2:] + [chunk_dicts[1]] if len(chunk_dicts) > 2 else chunk_dicts[:]
  context = "\n".join(f"[{n}] {i['text']}" for n, i in enumerate(reordered, 1))
  user = f"Фрагменты базы знаний:\n{context}\n\nВопрос: {query}"
  return [{"role": "system", "content": SYSTEM},
          {"role": "user", "content": user}]

def rag_answer(query, search_fn, k=5):
  """Полный RAG: retrieval -> промпт -> генерация."""
  ret = search_fn(query, k=k)
  msg = build_messages(query, ret)
  answer = generate(msg)
  return answer, [i['id'] for i in ret], [i['text'] for i in ret]

In [113]:
'''
5.2 Test different kinds of questions
'''
print('>>> exact queries')
for q in [x for x in queries if x["kind"] == "exact"][:3]:
  answer, ids, texts = rag_answer(q["query"], retrieve_rerank)
  print(f"\nВ: {q['query']}")
  print(f"   чанки: {ids}  ({[i[:34] for i in texts]})")
  print(f"   О: {answer}")

print('\n\n>>> semantic queries')
for q in [x for x in queries if x["kind"] == "semantic"][:3]:
  answer, ids, texts = rag_answer(q["query"], retrieve_rerank)
  print(f"\nВ: {q['query']}")
  print(f"   чанки: {ids}  ({[i[:34] for i in texts]})")
  print(f"   О: {answer}")

print('\n\n>>> multi queries')
for q in [x for x in queries if x["kind"] == "multi"][:3]:
  answer, ids, texts = rag_answer(q["query"], retrieve_rerank)
  print(f"\nВ: {q['query']}")
  print(f"   чанки: {ids}  ({[i[:34] for i in texts]})")
  print(f"   О: {answer}")

>>> exact queries

В: что означает ошибка P-102
   чанки: [92, 175, 95, 99, 106]  (['Ошибка P-102 появляется, когда чис', 'Ошибка P-112 появляется, когда акк', 'Ошибка P-110 появляется, когда про', 'Ошибка P-101 появляется, когда при', 'Если просмотр прерывается, плеер п'])
   О: Ошибка P-102 появляется, когда число одновременных просмотров на аккаунте превышает лимит тарифа. Сообщение показывается в плеере, просмотр при этом прерывается.

В: какая версия webOS или Tizen нужна для приложения на телевизоре
   чанки: [157, 183, 148, 42, 75]  (['Телевизоры Smart TV: Samsung (Tize', 'Приложение на телевизоре обновляет', 'Если вы не знаете, к какой платфор', 'Фильмы в онлайн-кинотеатрах переда', 'Если поиск в магазине телевизора н'])
   О: Для работы приложения на телевизорах нужны следующие версии операционных систем:
*   **Samsung (Tizen):** модели с 2018 года.
*   **LG (webOS):** 4.0 и новее.

В: обязателен ли HDCP 2.2 для просмотра в 4K
   чанки: [117, 116, 161, 122, 61]  (['Одного тари

In [114]:
'''
5.2 Test answer for gap-questions
'''
print('>>> gaps')
for q in [x for x in queries if x["kind"] == "gap"]:
  answer, ids, texts = rag_answer(q["query"], retrieve_rerank)
  print(f"\nВ: {q['query']}")
  print(f"   чанки: {ids}  ({[i[:34] for i in texts]})")
  print(f"   О: {answer}")

>>> gaps

В: есть ли подарочные сертификаты
   чанки: [0, 127, 54, 164, 94]  (['Промокод даёт скидку на подписку и', 'Сервис регулярно проводит акции: с', 'Всё в ней доступно по подписке, на', 'Напоминание о скором списании прих', 'Пользовательское соглашение (выдер'])
   О: В базе знаний ответа нет.

В: есть ли фильмы с аудиоописанием для слабовидящих
   чанки: [48, 78, 159, 171, 120]  (['У многих фильмов и сериалов есть н', 'Центральная колонка ставится строг', 'Субтитры включаются в плеере: во в', 'На состав каталога и на то, как фи', 'Если изображение идёт без звука ил'])
   О: В базе знаний ответа нет.

В: есть ли скидка для студентов
   чанки: [127, 0, 122, 107, 54]  (['Сервис регулярно проводит акции: с', 'Промокод даёт скидку на подписку и', 'Просмотр на четырёх устройствах од', 'ООО «Фильмоскоп» оказывает услуги ', 'Всё в ней доступно по подписке, на'])
   О: В базе знаний ответа нет.

В: когда выйдет продолжение сериала «Северный экспресс»
   чанки: [16, 163, 121, 164, 22]  (

# Этап 5. Генерация ответов - Report

Результат: RAG-система, которая принимает вопрос и возвращает ответ или корректный отказ.

For gap-queries the answer is unified: В базе знаний ответа нет - 9 из 10.
Проблемный ответ: Нет, скачивание для просмотра без интернета на телевизорах не работает.
- Ошибка модели, но соответствует "golden answer".

RAG is designed manually instead of using LangChain:
- could be fined-tuned on the later steps
- don't want add additional technological dependencies if possible

# Этап 6. Оценка качества и диагностика

## install
pip install pandas

In [115]:
'''
6.0 Load judge_cases
'''
import json

with open("6_judge_cases.json", encoding="utf-8") as f:
  judge_cases = json.load(f)

with open("2_chunks.json", encoding="utf-8") as f:
  chunks = json.load(f)
texts = [c["text"] for c in chunks]

In [116]:
'''
6.1 Judge Conf
'''
JUDGE_SYSTEM = (
    "Ты — строгий эксперт по оценке справочных ответов. Оценивай "
    "беспристрастно, длина ответа не является достоинством. "
    "Отвечай только JSON-объектом."
)

FAITH_PROMPT = """Оцени ВЕРНОСТЬ КОНТЕКСТУ (faithfulness):
следует ли каждое утверждение ответа из фрагментов.
Соответствие вопросу не оценивай — только происхождение фактов.

Шкала:
5 — все утверждения следуют из фрагментов;
4 — добавлено только безобидное обобщение;
3 — добавлена неподтверждённая деталь;
2 — часть утверждений выдумана или противоречит фрагментам;
1 — ключевые факты отсутствуют во фрагментах или искажены.
Ответ «В базе знаний ответа нет» ничего не утверждает и получает 5.

Фрагменты:
{context}

Ответ: {answer}

Верни JSON с полями reason и score."""

REL_PROMPT = """Оцени РЕЛЕВАНТНОСТЬ (answer relevance):
отвечает ли ответ на заданный вопрос по существу.
Правдивость не проверяй — только соответствие вопросу.

Шкала:
5 — прямой и полный ответ;
4 — ответ по существу с небольшими отступлениями;
3 — закрыта только часть вопроса;
2 — текст на нужную тему, но ответа нет;
1 — другая тема или отказ отвечать.

Вопрос: {question}

Ответ: {answer}

Верни JSON с полями reason и score."""

JUDGE_RESPONSE_FORMAT = {
    "type": "json_object",
    "schema": {
        "type": "object",
        "properties": {
            "reason": {"type": "string", "maxLength": 240},
            "score": {
                "type": "integer",
                "minimum": 1,
                "maximum": 5,
            },
        },
        "required": ["reason", "score"],
        "additionalProperties": False,
    },
}


def judge(prompt):
    """Возвращает оценку 1–5 и обоснование LLM-судьи."""
    response = client.chat.completions.create(
        model=LLM,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=200,
        response_format=JUDGE_RESPONSE_FORMAT,
        extra_body=NO_THINK,
    )
    value = json.loads(response.choices[0].message.content)
    score = value.get("score")
    reason = value.get("reason")
    if type(score) is not int or not 1 <= score <= 5:
        raise ValueError(f"Некорректная оценка судьи: {score!r}")
    if not isinstance(reason, str) or not reason.strip():
        raise ValueError("Судья не вернул обоснование")
    return score, reason.strip()

# кортеж «(оценка, обоснование)»
def judge_faithfulness(context, answer):
    """Оценивает верность ответа контексту."""
    prompt = FAITH_PROMPT.format(context=context, answer=answer)
    return judge(prompt)

# кортеж «(оценка, обоснование)»
def judge_relevance(question, answer):
    """Оценивает соответствие ответа вопросу."""
    prompt = REL_PROMPT.format(question=question, answer=answer)
    return judge(prompt)

def build_context(chunk_ids):
    """Склеивает чанки в нумерованный контекст."""
    return "\n".join(f"[{n}] {texts[i]}" for n, i in enumerate(chunk_ids, 1))

In [117]:
'''
6.3 Check Cases
'''
for case in judge_cases:
  print('>>> case: ', case['query'], case["context_ids"])
  context = build_context(case["context_ids"])
  print(judge_faithfulness(context, case["answer"]))
  print(judge_relevance(case["query"], case["answer"]))
  print('\n')

>>> case:  что означает ошибка P-102 [92, 175, 95, 99, 106]
(5, "Утверждение полностью соответствует фрагменту [1], где сказано: 'Ошибка P-102 появляется, когда число одновременных просмотров на аккаунте превышает лимит тарифа'.")
(5, 'Ответ напрямую и точно объясняет причину возникновения ошибки P-102, полностью соответствуя сути вопроса.')


>>> case:  вечером фильмы постоянно тормозят и крутится кружок загрузки, что делать [105, 190, 102, 17, 95]
(5, 'Все утверждения в ответе полностью соответствуют информации, содержащейся в фрагменте [1]. Утверждение о вечерней нагрузке сети и снижении качества, а также рекомендации по подключению кабелем и закрытию других приложений, взяты дословно из')
(5, 'Ответ полностью релевантен: он объясняет причину проблемы (нагрузка сети вечером) и предлагает конкретные решения (снижение качества в плеере, использование кабеля, закрытие других приложений).')


>>> case:  сколько устройств могут смотреть одновременно на Стандарте и что будет при превышени

In [76]:
'''
6.4 Определить порог - 3 или 4
'''
import numpy as np

# Четыре параллельных списка: по одному значению на каждый из 10 кейсов.
# judge_* — оценки нашего судьи, human_* — оценки асессора из corpus.json
judge_faith, judge_rel = [], []
human_faith, human_rel = [], []
# judged_rows — оценки обоих судей вместе с обоснованиями,
# чтобы потом напечатать проблемные ответы
judged_rows = []

for case in judge_cases:
    context = build_context(case["context_ids"])
    faith, faith_reason = judge_faithfulness(context, case["answer"])
    rel, rel_reason = judge_relevance(case["query"], case["answer"])
    judge_faith.append(faith)
    judge_rel.append(rel)
    human_faith.append(case["human_faithfulness"])
    human_rel.append(case["human_relevance"])
    judged_rows.append(
        {
            "id": case["id"],
            "faith": faith,
            "rel": rel,
            "faith_reason": faith_reason,
            "rel_reason": rel_reason,
        }
    )


def decision_agreement(scores, human_scores, threshold):
    """Считает совпадения решений по заданному порогу."""
    return sum(
        (score < threshold) == (human < threshold)
        for score, human in zip(scores, human_scores)
    )


print("Средний faithfulness:", round(float(np.mean(judge_faith)), 1))
print("Средний answer relevance:", round(float(np.mean(judge_rel)), 1))
print(
    "Точное совпадение faith:",
    sum(a == b for a, b in zip(judge_faith, human_faith)),
)
print(
    "Точное совпадение relevance:",
    sum(a == b for a, b in zip(judge_rel, human_rel)),
)
for threshold in (3, 4):
    print(
        "Порог ниже", threshold,
        "| faith", decision_agreement(judge_faith, human_faith, threshold),
        "| relevance", decision_agreement(judge_rel, human_rel, threshold),
    )

print("Проблемные ответы по рабочему порогу:")
for row in judged_rows:
    if row["faith"] < 4 or row["rel"] < 3:
        print(row)

Средний faithfulness: 5.0
Средний answer relevance: 3.5
Точное совпадение faith: 3
Точное совпадение relevance: 2
Порог ниже 3 | faith 3 | relevance 2
Порог ниже 4 | faith 3 | relevance 3
Проблемные ответы по рабочему порогу:
{'id': 4, 'faith': 5, 'rel': 1, 'faith_reason': 'Ответ «В базе знаний ответа нет» является корректным, так как ни одно из утверждений в ответе не содержит фактов, которые можно было бы проверить на соответствие предоставленным фрагментам. Фрагменты содержат информацию о промокодах, акциях', 'rel_reason': 'Ответ прямо указывает на отсутствие информации в базе знаний, что эквивалентно отказу отвечать на вопрос. Это соответствует шкале оценки 1 (другая тема или отказ отвечать).'}


## Threshold selected as 4
based on 'relevance 3'

In [18]:
'''
6.6
'''
import re

JUDGE_FAITH = (
    "Ты — строгий эксперт по оценке ответов. Оцени, насколько ОТВЕТ опирается "
    "только на ФРАГМЕНТЫ. 5 — каждое утверждение ответа есть во фрагментах; "
    "1 — ответ содержит факты, которых во фрагментах нет. Ответ «в базе знаний "
    "ответа нет» считай полностью опирающимся на фрагменты (5). "
    "Верни только одну цифру от 1 до 5.")

JUDGE_RELEV = (
    "Ты — строгий эксперт по оценке ответов. Оцени, отвечает ли ОТВЕТ по "
    "существу на ВОПРОС. 5 — прямой и полный ответ на заданный вопрос; "
    "1 — ответ не по теме или отказ отвечать. Верни только одну цифру от 1 до 5.")

JUDGE_CORRECT = (
    "Ты — строгий эксперт. Сравни ОТВЕТ с ЭТАЛОННЫМ ФАКТОМ. Верни 1, если ответ "
    "передаёт эталонный факт (формулировка может отличаться), и 0, если ответ "
    "противоречит эталону, неполон или отказывается отвечать. "
    "Верни только одну цифру: 1 или 0.")


def judge(question, context, answer, gold):
    """Оценивает answer для question по context и эталонному gold.

    Возвращает faithfulness, answer relevance и бинарную correctness.
    """
    ctx = "\n".join(f"[{i}] {c}" for i, c in enumerate(context, 1))
    faith = generate([{"role": "system", "content": JUDGE_FAITH},
                      {"role": "user", "content":
                       f"ФРАГМЕНТЫ:\n{ctx}\n\nОТВЕТ: {answer}"}], max_tokens=8)
    relev = generate([{"role": "system", "content": JUDGE_RELEV},
                      {"role": "user", "content":
                       f"ВОПРОС: {question}\n\nОТВЕТ: {answer}"}], max_tokens=8)
    correct = generate([{"role": "system", "content": JUDGE_CORRECT},
                        {"role": "user", "content":
                         f"ВОПРОС: {question}\nЭТАЛОННЫЙ ФАКТ: {gold}\n\n"
                         f"ОТВЕТ: {answer}"}], max_tokens=8)
    return (int(re.search(r"[1-5]", faith).group()),
            int(re.search(r"[1-5]", relev).group()),
            int(re.search(r"[01]", correct).group()))

In [ ]:
''' 6.6.1 Load queries & chunks'''
import json

with open("1_queries.json", encoding="utf-8") as f:
  queries = json.load(f)

with open("2_chunks.json", encoding="utf-8") as f:
  chunks = json.load(f)


def mrr(got_docs, relevant):
  """Обратный ранг первого релевантного раздела."""
  for rank, doc in enumerate(got_docs, start=1):
      if doc in relevant:
          return 1 / rank      # Первое место — 1, второе — 0,5, третье — 1/3
  return 0.0                   # Релевантных разделов в выдаче нет вовсе

In [39]:
'''
6.7 Judge - all queries
'''
rows = []
count = 0
for q in queries:
  count += 1
  print('>>> query #', count)

  answer, ctx, got_docs = rag_answer(q["query"], retrieve_rerank)
  faith, relev, correct = judge(q["query"], got_docs, answer, q["gold"])
  if correct < 1:
    print('\n>>> correctness problem:', correct, ', kind:', q["kind"],  
      '\n query:', q["query"], 
      '\n answer:', answer, 
      '\n gold:', q["gold"],
      '\n docs:', got_docs,
    )

  doc_ids = []
  for i in ctx:
    val = next((u for u in chunks if u.get('id') == i), None)
    doc_ids.append(val['doc_id'])

  row = {
      "kind": q["kind"],                # Тип запроса для разбивки по срезам
      "query": q["query"],
      # Метрики поиска считаем по разделам-источникам найденных чанков
      "precision@5": precision_at_k(doc_ids, q["relevant_docs"]),
      "recall@5": recall_at_k(doc_ids, set(q["relevant_docs"])),
      "MRR": mrr(doc_ids, q["relevant_docs"]),
      # Метрики генерации: их проставил судья
      "faithfulness": faith,
      "answer relevance": relev,
      "correctness": correct,
      "answer": answer,                 
      "ctx": ctx,                       
      "doc_ids": doc_ids,                       
  }
  rows.append(row)

  with open("6_judge_log.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(row, ensure_ascii=False) + "\n")

>>> # 1
>>> # 2
>>> # 3
>>> # 4

>>> correctness problem: 0 , kind: exact 
 query: как подключить саундбар по HDMI ARC 
 answer: В базе знаний ответа нет. 
 gold: подключить по HDMI ARC или оптическому кабелю; звук 5.1 доступен на тарифе «Максимум» 
 docs: ['Обозначение 5.1 описывает раскладку каналов: пять основных колонок вокруг зрителя и отдельный канал низких частот. За счёт этого шаги, дождь или проезжающая машина слышны с той стороны, где они происходят в кадре. Не путайте многоканальную дорожку с виртуальным объёмным звуком, который умеют делать саундбары и наушники: там обычная стереодорожка расширяется самим устройством. К тому, что приходит из сервиса, эта обработка отношения не имеет. Объёмный звук 5.1 доступен только на тарифе «Максимум» и только для фильмов и сериалов, у которых есть дорожка 5.1 — они отмечены в карточке значком «5.1». Чтобы услышать объёмный звук, нужна соответствующая аудиосистема: домашний кинотеатр, ресивер или саундбар с поддержкой многоканального зву

In [43]:
'''
6.8.1 Retrieval Stat
'''
import json
import pandas as pd

with open("6_judge_log.jsonl", "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

# Широкий вывод: иначе pandas свернёт часть столбцов в многоточие
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", None)

print(">>> Queries:", len(data))

filtered = [i for i in data if i['kind'] != 'gap']
report = pd.DataFrame(filtered)
METRICS = ["precision@5", "recall@5", "MRR"]
print(report.groupby("kind")[METRICS].mean().round(3))   # Срезы по полю kind
print('\nall queries (except gaps):\n', report[METRICS].mean().round(3))   # Средние по всему набору

gaps = [i for i in data if i['kind'] == 'gap']
report = pd.DataFrame(gaps)
METRICS = ["precision@5", "recall@5", "MRR"]
print('\ngaps:\n', report[METRICS].mean().round(3))   # Средние по всему набору

>>> Queries: 66
          precision@5  recall@5    MRR
kind                                  
exact           0.293     1.000  1.000
multi           0.475     0.875  0.859
semantic        0.240     0.960  0.708

all queries (except gaps):
 precision@5    0.321
recall@5       0.946
MRR            0.829
dtype: float64

gaps:
 precision@5    0.0
recall@5       0.0
MRR            0.0
dtype: float64


In [47]:
'''
6.8.2 Final Stat
'''
report = pd.DataFrame(data)
METRICS = ["precision@5", "recall@5", "MRR", "faithfulness", "answer relevance", "correctness"]

print("Total metrics - all queries:")
print(report[METRICS].mean().round(3))          # Средние по всему набору

print("\nПо типам запросов:")
print(report.groupby("kind")[METRICS].mean().round(3))   # Срезы по полю kind

Total metrics - all queries:
precision@5         0.273
recall@5            0.803
MRR                 0.704
faithfulness        4.924
answer relevance    4.182
correctness         0.879
dtype: float64

По типам запросов:
          precision@5  recall@5    MRR  faithfulness  answer relevance  correctness
kind                                                                               
exact           0.293     1.000  1.000         5.000             4.733        0.933
gap             0.000     0.000  0.000         5.000             1.400        1.000
multi           0.475     0.875  0.859         4.688             4.938        0.875
semantic        0.240     0.960  0.708         5.000             4.480        0.800



## 6.8.3 Manual Problem Analysis

1. exact: как подключить саундбар по HDMI ARC 
 answer: В базе знаний ответа нет. 
 gold: подключить по HDMI ARC или оптическому кабелю; звук 5.1 доступен на тарифе «Максимум» 
 docs: ['Обозначение 5.1 описывает раскладку каналов: пять основных колонок вокруг зрителя и отдельный канал низких частот. За счёт этого шаги, дождь или проезжающая машина слышны с той стороны, где они происходят в кадре. Не путайте многоканальную дорожку с виртуальным объёмным звуком, который умеют делать саундбары и наушники: там обычная стереодорожка расширяется самим устройством. К тому, что приходит из сервиса, эта обработка отношения не имеет. Объёмный звук 5.1 доступен только на тарифе «Максимум» и только для фильмов и сериалов, у которых есть дорожка 5.1 — они отмечены в карточке значком «5.1». Чтобы услышать объёмный звук, нужна соответствующая аудиосистема: домашний кинотеатр, ресивер или саундбар с поддержкой многоканального звука, подключённые к телевизору по HDMI ARC или оптическому кабелю.', 'Если при воспроизведении звук идет, а изображения нет — экран остается черным, — дело почти всегда в HDMI-подключении и защите HDCP. Что попробовать: переподключите HDMI-кабель с обеих сторон; подключите устройство в другой HDMI-порт телевизора; замените кабель — старые кабели не поддерживают нужную версию HDCP; если приставка подключена через ресивер или саундбар, подключите ее напрямую к телевизору. После каждого изменения перезапускайте воспроизведение.', 'Объёмный звук дома собирают двумя способами, и выбор между ними — это выбор между простотой и честностью звучания. Саундбар — одна длинная колонка под телевизором, часто в комплекте с беспроводным сабвуфером. Занимает мало места, ставится за несколько минут, объём создаёт направленными излучателями и отражением звука от стен и потолка. Платить за это приходится условным тылом: звук позади зрителя скорее обозначен, чем действительно есть. Домашний кинотеатр на ресивере — вариант посложнее. Ресивер это усилитель с коммутацией: к нему подключаются колонки и телевизор, он же раскладывает многоканальную дорожку по каналам. Проводов и места нужно больше, зато каждый канал играет своей колонкой. Расстановка комплекта 5.1 подчиняется простым правилам.', 'Центральная колонка ставится строго под экраном или над ним — на неё приходятся диалоги, и промахи в её положении слышны сильнее всего. Фронтальные ставят по краям экрана примерно на высоте уха сидящего зрителя, тыловые — чуть позади дивана и немного выше головы. Сабвуфер можно поставить почти где угодно, но в углу он гудит сильнее, чем у стены. Какие фильмы идут с многоканальной дорожкой и что нужно, чтобы услышать её в сервисе, описано в отдельном разделе справки.', 'Субтитры включаются в плеере: во время просмотра откройте меню «Субтитры» и выберите язык. Там же субтитры отключаются. Если пункта «Субтитры» нет, значит, для этого фильма или серии субтитры не подготовлены — они доступны не для всего каталога. Если субтитры включены, но не отображаются, перезапустите воспроизведение; на телевизорах также помогает обновление приложения до свежей версии.']

! wrong answer

- retrieval:  +
- reranking:  +
- generation: -


2. semantic: включаю кино — на экране всё идёт, а из колонок тишина 
 answer: В базе знаний ответа нет. 
 gold: начать с меню аудиодорожек в плеере и проверить громкость и настройки звука устройства 
 docs: ['Если изображение идёт без звука или звук отстаёт от картинки, начните с плеера: откройте меню аудиодорожек и убедитесь, что выбрана нужная дорожка, а громкость в плеере не убавлена до нуля. Дальше проверьте устройство: громкость и режим звука телевизора, подключённые колонки или наушники. Помогает и простой перезапуск: закройте приложение полностью и откройте фильм заново. Рассинхронизация звука после перемотки исчезает после паузы на несколько секунд или перезапуска серии.', 'Экономичные режимы яркости и лишняя обработка изображения способны испортить даже корректный 4K-поток, поэтому для кино на телевизоре обычно выбирают отдельный режим изображения. Если изображения нет вовсе — экран остаётся чёрным, хотя звук идёт, — это уже не ограничение качества, а отдельная неполадка. Ей посвящён свой раздел в справке об ошибках воспроизведения. Сначала стоит убедиться, что фильм вообще запускается, и только потом разбираться с качеством.', 'Встроенные динамики телевизора делают тонкими вместе с корпусом, поэтому внешнюю акустику к нему подключают даже те, кто не собирался заниматься звуком всерьёз. Способ подключения зависит от того, какие разъёмы есть у телевизора и у колонок: у одних моделей это выход на наушники, у других — отдельные выходы для внешней аудиосистемы, у третьих всё держится на беспроводной паре. Что есть на вашей модели, написано в инструкции к телевизору. Беспроводное подключение удобнее проводного, но у него есть цена — задержка. Звук приходит на колонку чуть позже картинки, и на разговорных сценах это заметно: губы уже сомкнулись, а фраза ещё идёт. В настройках звука телевизора обычно есть пункт для сдвига звука относительно изображения — его двигают на слух.', 'Центральная колонка ставится строго под экраном или над ним — на неё приходятся диалоги, и промахи в её положении слышны сильнее всего. Фронтальные ставят по краям экрана примерно на высоте уха сидящего зрителя, тыловые — чуть позади дивана и немного выше головы. Сабвуфер можно поставить почти где угодно, но в углу он гудит сильнее, чем у стены. Какие фильмы идут с многоканальной дорожкой и что нужно, чтобы услышать её в сервисе, описано в отдельном разделе справки.', 'Если при воспроизведении звук идет, а изображения нет — экран остается черным, — дело почти всегда в HDMI-подключении и защите HDCP. Что попробовать: переподключите HDMI-кабель с обеих сторон; подключите устройство в другой HDMI-порт телевизора; замените кабель — старые кабели не поддерживают нужную версию HDCP; если приставка подключена через ресивер или саундбар, подключите ее напрямую к телевизору. После каждого изменения перезапускайте воспроизведение.']

- retrieval:  +
- reranking:  +
- generation: -


3. semantic: подключил колонки, но объёмного звука нет — идёт обычный стерео 
 answer: На встроенных динамиках телевизора дорожка 5.1 автоматически сводится в стерео. Чтобы многоканальный сигнал уходил на внешнюю систему, в настройках звука телевизора выбирают вывод на подключённое устройство и отключают встроенные динамики. Названия пунктов у разных производителей отличаются, но смысл один. 
 gold: нужен тариф «Максимум», дорожка 5.1 у фильма и подключение по HDMI ARC или оптике; на динамиках телевизора звук сводится в стерео 
 docs: ['Обозначение 5.1 описывает раскладку каналов: пять основных колонок вокруг зрителя и отдельный канал низких частот. За счёт этого шаги, дождь или проезжающая машина слышны с той стороны, где они происходят в кадре. Не путайте многоканальную дорожку с виртуальным объёмным звуком, который умеют делать саундбары и наушники: там обычная стереодорожка расширяется самим устройством. К тому, что приходит из сервиса, эта обработка отношения не имеет. Объёмный звук 5.1 доступен только на тарифе «Максимум» и только для фильмов и сериалов, у которых есть дорожка 5.1 — они отмечены в карточке значком «5.1». Чтобы услышать объёмный звук, нужна соответствующая аудиосистема: домашний кинотеатр, ресивер или саундбар с поддержкой многоканального звука, подключённые к телевизору по HDMI ARC или оптическому кабелю.', 'На встроенных динамиках телевизора дорожка 5.1 автоматически сводится в стерео. Чтобы многоканальный сигнал уходил на внешнюю систему, в настройках звука телевизора выбирают вывод на подключённое устройство и отключают встроенные динамики. Названия пунктов у разных производителей отличаются, но смысл один. Выбранная звуковая дорожка запоминается в настройках профиля, поэтому у одного члена семьи может быть включена одна дорожка, а у другого — другая. Настройки одного профиля на остальные не влияют. Если фильм открыли в другом профиле, дорожку придётся выбрать там заново. Если аудиосистема поддерживает объёмный звук, а фильм выпущен только со стереодорожкой, сервис ничего не добавит: расширение стерео на несколько колонок выполняет сама аудиосистема, и качество такой обработки зависит только от неё.', 'Объёмный звук дома собирают двумя способами, и выбор между ними — это выбор между простотой и честностью звучания. Саундбар — одна длинная колонка под телевизором, часто в комплекте с беспроводным сабвуфером. Занимает мало места, ставится за несколько минут, объём создаёт направленными излучателями и отражением звука от стен и потолка. Платить за это приходится условным тылом: звук позади зрителя скорее обозначен, чем действительно есть. Домашний кинотеатр на ресивере — вариант посложнее. Ресивер это усилитель с коммутацией: к нему подключаются колонки и телевизор, он же раскладывает многоканальную дорожку по каналам. Проводов и места нужно больше, зато каждый канал играет своей колонкой. Расстановка комплекта 5.1 подчиняется простым правилам.', 'Если изображение идёт без звука или звук отстаёт от картинки, начните с плеера: откройте меню аудиодорожек и убедитесь, что выбрана нужная дорожка, а громкость в плеере не убавлена до нуля. Дальше проверьте устройство: громкость и режим звука телевизора, подключённые колонки или наушники. Помогает и простой перезапуск: закройте приложение полностью и откройте фильм заново. Рассинхронизация звука после перемотки исчезает после паузы на несколько секунд или перезапуска серии.', 'Встроенные динамики телевизора делают тонкими вместе с корпусом, поэтому внешнюю акустику к нему подключают даже те, кто не собирался заниматься звуком всерьёз. Способ подключения зависит от того, какие разъёмы есть у телевизора и у колонок: у одних моделей это выход на наушники, у других — отдельные выходы для внешней аудиосистемы, у третьих всё держится на беспроводной паре. Что есть на вашей модели, написано в инструкции к телевизору. Беспроводное подключение удобнее проводного, но у него есть цена — задержка. Звук приходит на колонку чуть позже картинки, и на разговорных сценах это заметно: губы уже сомкнулись, а фраза ещё идёт. В настройках звука телевизора обычно есть пункт для сдвига звука относительно изображения — его двигают на слух.']

- retrieval:  +
- reranking:  +
- generation: -


4. semantic: старый телевизор не запускает фильмы, пишет что-то про защиту содержимого 
 answer: Если фильм не запускается и плеер показывает сообщение про защиту содержимого, разбираться нужно по коду ошибки: в справке об ошибках воспроизведения для каждого кода описан свой порядок действий. 
 gold: устройство не поддерживает DRM; помогает обновление прошивки, иначе нужно другое устройство 
 docs: ['Если фильм не запускается и плеер показывает сообщение про защиту содержимого, разбираться нужно по коду ошибки: в справке об ошибках воспроизведения для каждого кода описан свой порядок действий.', 'Ошибка P-104 появляется, когда устройство не поддерживает защиту контента (DRM), которую использует «Фильмоскоп». Сообщение показывается в плеере, просмотр при этом прерывается. Перезапустите приложение и попробуйте снова. Проверьте, что установлена последняя версия приложения и на устройстве есть свободное место. Обновите прошивку устройства в настройках телевизора или приставки: поддержку актуальной версии защиты производители добавляют именно в обновлениях прошивки. Если прошивка последняя, а ошибка осталась, запустите фильм на другом устройстве из списка поддерживаемых. Если ошибка повторяется, запишите её код и напишите в поддержку через чат в приложении. Укажите время появления ошибки: по нему поддержка найдёт запись в журналах и разберёт случай быстрее.', 'Фильмы в онлайн-кинотеатрах передаются в зашифрованном виде. Это условие правообладателей: без защиты студии не отдают картины в подписку. Устроено это так. Файл на серверах зашифрован, ключ выдаётся устройству отдельно и только на время сеанса. Расшифровка происходит внутри устройства, в отдельном блоке, к которому нет доступа ни у приложения, ни у остальной системы. Требования у правообладателей разные, и для потоков высокого качества условия строже, чем для обычных. Поэтому один и тот же телевизор с одним и тем же приложением может показывать одно и не показывать другое. Отсюда и разница между устройствами. Защита реализована в железе и в системе устройства, а не в приложении, так что установка более свежей версии приложения ничего не меняет. Отвечает за неё производитель устройства: возможности меняются вместе с системными обновлениями, а когда поддержка модели заканчивается, остаются такими, какими были.', 'Запуск фильма — это цепочка: приложение обращается к серверам сервиса, получает видео, расшифровывает его и выводит на экран. Ошибка появляется тогда, когда одно из звеньев не отработало, и её код говорит, какое именно. Причины делятся на четыре группы. Сеть: соединение пропало или оказалось слишком медленным для выбранного качества. Устройство: не хватает памяти, стоит старая версия приложения или прошивки, мешают настройки браузера. Права на контент: у фильмов и сериалов свои условия показа, и часть каталога открыта не для всех. Нагрузка на сервис: серверы отвечают медленно или недоступны. Знать группу полезно, но для решения этого мало: в каждой из них несколько разных ситуаций, и шаги у них разные. Поэтому смотреть нужно на код с экрана и открывать статью справки, посвящённую именно ему.', 'Если приложение на телевизоре закрывается само или зависает при запуске, действуйте по шагам. Перезапустите приложение полностью, а затем — сам телевизор, выключив его из розетки на минуту. Очистите кеш приложения в настройках телевизора (раздел «Приложения»). Проверьте, есть ли обновление приложения и прошивки. Если не помогло — удалите приложение и установите заново из магазина. Частые вылеты на старых моделях связаны с нехваткой памяти: закройте другие приложения, работающие в фоне.']

! best chunk is 2nd 

- retrieval:  +
- reranking:  -
- generation: -



5. semantic: хочу убрать из ленты фильмы, которые уже смотрел 
 answer: В базе знаний ответа нет. 
 gold: удалить записи из истории просмотров в настройках профиля; история хранится отдельно для каждого профиля 
 docs: ['Лента на главном экране собирается не из воздуха: за ней стоят записи о том, что вы смотрели. Сервис запоминает, какие фильмы и серии вы включали, докуда досмотрели, что оценили «Нравится» и «Не нравится» и что отложили в «Моё». Отдельно сохраняются технические сведения — устройство, качество, время начала просмотра: по ним разбирают жалобы на воспроизведение. Записи нужны для трёх вещей. Первая — рекомендации: чем больше сервис знает о вкусе, тем меньше в подборках случайного. Вторая — незаконченные фильмы: без записи о просмотре сервис не отличит начатое от неначатого. Третья — разбор обращений в поддержку, когда нужно понять, что происходило на устройстве. Записи привязаны к профилю, а не к аккаунту: у каждого зрителя своя картина просмотров, и на чужие подборки она не влияет. Профиль, созданный недавно, сервису пока не известен — первые дни подборки в нём общие для всех.', 'Кнопка «В Моё» в карточке фильма или сериала добавляет его в личный список — раздел «Моё» на главном экране. Список синхронизируется между всеми устройствами: добавили на телефоне — увидите на телевизоре. У каждого профиля список свой. Убрать фильм из списка можно той же кнопкой в карточке или прямо в разделе «Моё». Раздел «Моё» также хранит купленные фильмы — они отмечены отдельной полкой «Покупки».', 'История просмотров хранится отдельно для каждого профиля: раздел «История» в настройках профиля. Из истории можно удалить отдельную запись — наведите на неё и нажмите «Убрать из истории». Вместе с записью из «Продолжить просмотр» пропадёт и позиция воспроизведения. Удаление записей влияет на рекомендации: сервис перестаёт учитывать удалённый контент при подборе похожего. Полностью отключить ведение истории нельзя — без неё не работают «Продолжить просмотр» и рекомендации.', 'Профиль — это отдельное пространство внутри одного аккаунта: у каждого своя лента и свои рекомендации. Фильмы, которые смотрит один человек, не попадают в подборки другого, даже если оба заходят с одного устройства. Переключиться между профилями можно в любой момент: экран выбора профиля показывается при запуске приложения, а из меню профиля к нему возвращаются во время работы. Перезаходить в аккаунт для этого не нужно. В одном аккаунте можно создать до 5 профилей — по одному на каждого члена семьи. У каждого профиля свои: история просмотров и раздел «Продолжить просмотр»; подборки и рекомендации; список «Моё»; настройки языка и субтитров. Профили не увеличивают число одновременных просмотров — оно определяется тарифом. Создать, переименовать или удалить профиль можно на экране выбора профиля. Профиль можно удалить, но вместе с ним удаляются его история просмотров, список «Моё» и накопленные рекомендации — восстановить их потом нельзя.', 'Удалить аккаунт можно в настройках: раздел «Личные данные» - «Удалить аккаунт». Перед удалением отмените подписку — иначе списания продолжатся до конца оформленного периода. После подачи заявки аккаунт замораживается на 30 дней: в течение этого срока удаление можно отменить, просто войдя в аккаунт. Через 30 дней аккаунт и его данные — профили, история просмотров, список «Моё», привязанные устройства — удаляются безвозвратно. Купленные фильмы при удалении аккаунта также теряются.']

! best chunk is 3rd 

- retrieval:  +
- reranking:  -
- generation: -


6. semantic: в приложении всё включено, а на телефон ничего не приходит 
 answer: В базе знаний ответа нет. 
 gold: проверить системные настройки телефона: разрешены ли уведомления приложению и не включён ли режим «Не беспокоить»; после переустановки разрешение выдают заново 
 docs: ['Если на экране появилась ошибка, пройдите базовые шаги — они решают большинство разовых сбоев: 1. Перезапустите фильм, затем приложение. 2. Перезагрузите устройство. 3. Проверьте, что приложение обновлено до свежей версии. 4. Проверьте, что интернет работает на других сайтах и приложениях. 5. Загляните на страницу статуса сервиса: возможно, идут технические работы. Если ошибка осталась, найдите в справке статью по её коду — там будут шаги для конкретной ситуации.', 'Вход с устройства, которым в аккаунте раньше не пользовались, сервис отмечает уведомлением. Такие уведомления относятся к сервисным и не отключаются: они существуют ровно затем, чтобы чужой вход не остался незамеченным. Приходит обычно два сообщения: письмо на привязанную почту и пуш-уведомление на телефон, где установлено приложение. В тексте указаны дата и время входа, тип устройства и приложение, из которого он выполнен, — например «телевизор, приложение "Фильмоскоп"» или «браузер на компьютере». Сообщение о новом устройстве иногда приходит и на знакомое. После переустановки приложения, обновления телевизора или очистки данных браузера сервис видит устройство как незнакомое и предупреждает о входе заново. Само по себе это не признак чужого доступа. Если вход ваш, делать ничего не нужно: уведомление просто фиксирует событие.', 'Приложение на телевизоре обновляется само, если в магазине включено автообновление. Обычно оно включено по умолчанию, а переключатель лежит в настройках магазина, а не в настройках приложения. Работает оно в фоне: телевизор должен быть включён или хотя бы стоять в дежурном режиме с доступом к сети. Поэтому на телевизоре, который включают на час вечером, свежая версия появляется позже, чем на телефоне. Обновить вручную можно из того же магазина. Откройте раздел с установленными приложениями — рядом с теми, для которых вышла новая сборка, стоит кнопка обновления. Если кнопки нет, а приложение ведёт себя странно, помогает переустановка: удалите его и поставьте заново. Аккаунт и история просмотров хранятся на сервере и при переустановке не теряются. Номер установленной версии виден в самом приложении, в конце списка настроек. Его же спрашивает поддержка: по версии сразу понятно, есть ли на устройстве уже вышедшее исправление.', 'Чаще всего вход не проходит не из-за поломки, а из-за мелочи, которую легко не заметить. Опечатка в логине. Адрес почты и номер телефона вводятся целиком и без пробелов; лишний пробел в конце строки приложение не показывает, но в аккаунт с ним не пускает. Стоит проверить и раскладку клавиатуры, и Caps Lock: пароль чувствителен к регистру. Два аккаунта. Если когда-то вы регистрировались по номеру телефона, а потом ещё раз по почте — это разные аккаунты с разными подписками. Вход проходит, но внутри пусто: чужая лента, нет истории, подписка «не найдена». Сохранённый пароль. Браузер или менеджер паролей подставляет старое значение, если пароль меняли на другом устройстве. Введите его вручную, а не выбором из списка. Старая версия приложения. На телевизорах и приставках обновления иногда не ставятся сами, и в старой сборке экран входа может не открыться или зависнуть на загрузке. Помогает обновление приложения из магазина устройства.', 'Скачивание фильмов и сериалов срывается чаще, чем обычный просмотр: загрузка идёт долго, и за это время успевает измениться то, от чего она зависит. Приложение показывает сообщение об ошибке и останавливает скачивание, а начатый файл остаётся в разделе «Загрузки» незавершённым. Перед большой загрузкой посмотрите, сколько места осталось на устройстве, и подключитесь к стабильной сети: на мобильном интернете скачивание обрывается заметно чаще, чем по Wi-Fi. Не закрывайте приложение полностью, пока идёт загрузка, и не переводите устройство в режим экономии энергии — в нём фоновые задачи останавливаются. Если загрузка встала и не возобновляется, удалите её из списка и запустите скачивание заново — это надёжнее, чем ждать продолжения. Когда приложение показывает код ошибки, разбирать сбой по общим советам не нужно: найдите в справке статью с этим кодом, там описаны причина и точные шаги.']
 
! wrong answer

- retrieval:  -
- reranking:  -
- generation: -


7. multi: чем Стандарт отличается от Максимума и потеряю ли оплаченные дни при переходе 
 answer: Тариф «Стандарт» отличается от тарифа «Максимум» следующими параметрами (согласно фрагменту [2]):
*   **Количество устройств:** «Стандарт» позволяет просмотр на двух устройствах одновременно, тогда как «Максимум» (по контексту тарифов) подразумевает большее количество одновременных просмотров.
*   **Качество картинки:** «Стандарт» обеспечивает качество до Full HD (1080p), а «Максимум» предлагает более высокое качество (точное значение в фрагментах не указано, но подразумевается выше, чем у «Стандарта»).
*   **Функционал:** В обоих тарифах доступно скачивание на мобильные устройства.
При перех 
 gold: «Максимум» дороже и даёт 4K HDR, звук 5.1 и четыре одновременных просмотра; при переходе вверх неиспользованные дни пересчитываются, они не сгорают 
 docs: ['Перед переходом стоит сравнить условия: от тарифа зависят число одновременных просмотров, максимальное качество картинки и доступ к скачиванию на мобильные устройства. Полный список условий приведён в разделе о тарифах «Фильмоскопа». Смена тарифа не затрагивает содержимое аккаунта. Профили, история просмотров, список «Моё» и привязанные устройства остаются на месте — меняются только условия подписки. Заново входить в аккаунт на устройствах не нужно. Перейти на другой тариф можно в разделе «Подписка» настроек аккаунта. При переходе на более дорогой тариф изменения вступают в силу сразу: сервис пересчитает неиспользованные дни текущего тарифа и зачтёт их в счёт нового, поэтому оплаченные дни не сгорают. При переходе на более дешёвый тариф текущий тариф действует до конца оплаченного периода, а новый включится со следующего списания. Доплат и возвратов при таком переходе нет.', 'В «Фильмоскопе» три тарифа: «Мобильный», «Стандарт» и «Максимум». Они отличаются числом одновременных просмотров, максимальным качеством картинки и набором функций — точные условия каждого собраны в статье о тарифах. Здесь мы разберём, на какие вопросы стоит ответить до того, как открывать таблицу. Сколько человек смотрит одновременно. Считайте не тех, у кого есть доступ к аккаунту, а тех, кто включает кино в одно и то же время: в будни вечером это обычно кто-то один, а в выходные — вся семья сразу по разным комнатам. На каком экране вы смотрите. То, что незаметно на телефоне в дороге, на большом телевизоре в гостиной бросается в глаза сразу. Если дома стоит такой телевизор и вы цените детализованную картинку, посмотрите в таблице тарифов, на каких из них доступно максимальное качество. Тем, кто чаще включает кино фоном, эта разница вряд ли окупится. Смотрите ли вы без интернета.', 'Тариф выбирается при оформлении подписки и в любой момент виден в настройках аккаунта, в разделе «Подписка»: там указаны название текущего тарифа, дата следующего списания и способ оплаты. Там же тариф меняется на другой — порядок перехода описан в отдельном разделе справки. От тарифа зависит только то, как вы смотрите: сколько экранов работает одновременно, до какого качества поднимается картинка и доступно ли скачивание на мобильные устройства. Число привязанных к аккаунту устройств и число профилей от тарифа не зависят — эти лимиты одинаковы для всех подписок. Сервис работает по подписке. Доступны три тарифа: «Мобильный» — 199 ₽ в месяц. Просмотр на одном устройстве, только телефон или планшет. Качество до 720p. «Стандарт» — 399 ₽ в месяц. Просмотр на двух устройствах одновременно, качество до Full HD (1080p), доступно скачивание на мобильные устройства. «Максимум» — 599 ₽ в месяц.', 'Порядок перехода и то, что происходит с уже оплаченными днями, описаны в разделе о смене тарифа; отдельно оформлять новую годовую подписку для этого не нужно. Отмена годовой подписки и правила возврата последнего платежа не отличаются от помесячных — им посвящён отдельный раздел справки. Оформляется отмена там же, в разделе «Подписка», где подключалась годовая оплата. Дата, до которой оплачен год, видна в настройках подписки. Напоминание приходит на почту, привязанную к аккаунту, — если почта изменилась, поменяйте её заранее, иначе письмо уйдёт на старый адрес. Проверить и изменить адрес можно в настройках аккаунта.', 'Пробный период Начинается всё с обычного оформления: зарегистрируйте аккаунт, выберите тариф и укажите способ оплаты. Карта нужна сразу — она понадобится для будущих списаний, но во время бесплатного доступа деньги с неё не снимаются. Сервис в это время работает без ограничений: доступен весь каталог фильмов и сериалов без рекламы, профили, подборки и остальные функции выбранного тарифа. Включать что-либо отдельно не нужно. Смотреть можно на всех устройствах, которые поддерживает выбранный тариф. Новым пользователям доступен пробный период — 14 дней бесплатного доступа к тарифу «Стандарт». Пробный период подключается при первом оформлении подписки и доступен один раз на аккаунт. Чтобы пробный период не перешёл в платную подписку, отмените её в настройках до окончания 14 дней — деньги не спишутся. Если этого не сделать, по окончании пробного периода подписка продлится автоматически по цене выбранного тарифа.']

- retrieval:  +
- reranking:  +
- generation: -


8. multi: скачал сериал в дорогу, а место на телефоне кончилось и загрузка сорвалась 
 answer: Скачивание фильмов и сериалов срывается чаще, чем обычный просмотр: загрузка идёт долго, и за это время успевает измениться то, от чего она зависит. Приложение показывает сообщение об ошибке и останавливает скачивание, а начатый файл остаётся в разделе «Загрузки» незавершённым. Перед большой загрузкой посмотрите, сколько места осталось на устройстве, и подключитесь к стабильной сети: на мобильном интернете скачивание обрывается заметно чаще, чем по Wi-Fi. Не закрывайте приложение полностью, пока идёт загрузка, и не переводите устройство в режим экономии энергии — в нём фоновые задачи останавливаются. Если загрузка встала и не возобновляется, удал 
 gold: ошибка P-109 означает нехватку места; нужно освободить память, при этом на устройстве хранится до 20 загрузок 
 docs: ['Скачивание фильмов и сериалов срывается чаще, чем обычный просмотр: загрузка идёт долго, и за это время успевает измениться то, от чего она зависит. Приложение показывает сообщение об ошибке и останавливает скачивание, а начатый файл остаётся в разделе «Загрузки» незавершённым. Перед большой загрузкой посмотрите, сколько места осталось на устройстве, и подключитесь к стабильной сети: на мобильном интернете скачивание обрывается заметно чаще, чем по Wi-Fi. Не закрывайте приложение полностью, пока идёт загрузка, и не переводите устройство в режим экономии энергии — в нём фоновые задачи останавливаются. Если загрузка встала и не возобновляется, удалите её из списка и запустите скачивание заново — это надёжнее, чем ждать продолжения. Когда приложение показывает код ошибки, разбирать сбой по общим советам не нужно: найдите в справке статью с этим кодом, там описаны причина и точные шаги.', 'Офлайн-просмотр — это фильмы и серии, скачанные заранее, чтобы открываться без интернета. Смысл в нём появляется там, где сети или нет вовсе, или она есть, но для видео не годится. Дорога — главный сценарий. В самолёте интернета обычно нет, в поезде он пропадает вместе со связью, в метро исчезает между станциями, на даче ловит одну палочку. Онлайн-просмотр в таких условиях превращается в чередование картинки и кружка загрузки. Скачанный файл этим не занят: он уже лежит на устройстве целиком, и дорога на него никак не влияет. Второй повод — мобильный трафик. Часа видео в высоком качестве хватает, чтобы заметно проредить пакет оператора. Скачать серии дома по домашнему Wi-Fi выходит спокойнее, чем везти их в поездку по мобильной сети.', 'Ошибка P-113 появляется, когда файл загрузки повреждён. Сообщение показывается в плеере, просмотр при этом прерывается. Перезапустите приложение и попробуйте снова. Проверьте, что установлена последняя версия приложения и на устройстве есть свободное место. Удалите повреждённую загрузку в разделе «Загрузки»: восстановить такой файл нельзя. Скачайте фильм или серию заново и дождитесь, пока загрузка дойдёт до конца. Если ошибка повторяется, запишите её код и напишите в поддержку через чат в приложении. Укажите время появления ошибки: по нему поддержка найдёт запись в журналах и разберёт случай быстрее.', 'Ограничения: **скачивать можно не весь каталог:** - у части контента нет прав на офлайн-просмотр - на одном устройстве может храниться до 20 загрузок - скачанный файл доступен до 30 дней - при подключении к интернету срок продлевается автоматически Загрузка привязана к устройству, на котором сделана: на телефоне и на планшете это разные загрузки, даже если аккаунт один. Перенести скачанный файл с одного устройства на другое нельзя — на втором его придётся скачать заново. Несколько серий можно поставить в очередь: они скачиваются по одной, а прогресс виден в разделе «Загрузки». Пока приложение свёрнуто, загрузка продолжается, но если закрыть его полностью, очередь встанет до следующего запуска. В настройках приложения есть отдельный переключатель для скачивания по мобильной сети. Если его выключить, загрузки будут начинаться только при подключении к Wi-Fi — так фильмы не израсходуют мобильный трафик незаметно.', 'Ошибка P-111 появляется, когда истёк срок действия скачанного файла. Сообщение показывается в плеере, просмотр при этом прерывается. Перезапустите приложение и попробуйте снова. Проверьте, что установлена последняя версия приложения и на устройстве есть свободное место. Подключитесь к интернету и откройте раздел «Загрузки»: при подключении срок действия файла продлевается. Если продлить не удалось, удалите файл и скачайте фильм или серию заново. Если ошибка повторяется, запишите её код и напишите в поддержку через чат в приложении. Укажите время появления ошибки: по нему поддержка найдёт запись в журналах и разберёт случай быстрее.']

! retrieval + reranking - chunk 1 from 2, 4th 

- retrieval:  +/-
- reranking:  -
- generation: -

? iterative RAG

In [66]:
'''
6.9 golden run
'''
# def build_messages(query, chunk_dicts):
#   """Собирает промпт: system-инструкция + нумерованный контекст + вопрос."""
#   reordered = [chunk_dicts[0]] + chunk_dicts[2:] + [chunk_dicts[1]] if len(chunk_dicts) > 2 else chunk_dicts[:]
#   context = "\n".join(f"[{n}] {i['text']}" for n, i in enumerate(reordered, 1))
#   user = f"Фрагменты базы знаний:\n{context}\n\nВопрос: {query}"
#   return [{"role": "system", "content": SYSTEM},
#           {"role": "user", "content": user}]

def golden_answer(query):
  """Полный RAG: retrieval -> промпт -> генерация."""
  doc_ids = query['relevant_docs']
  docs = [с for с in chunks if с['doc_id'] in doc_ids]
  # print('>>> doc_ids', doc_ids, docs)
  msg = build_messages(query['query'], docs)
  answer = generate(msg)
  return answer, [i['id'] for i in docs], [i['text'] for i in docs], doc_ids


from pathlib import Path
Path("6_judge_golden_log.jsonl").unlink(missing_ok=True)

rows = []
for q in queries[:]:
  answer, ctx, got_docs, doc_ids = golden_answer(q)
  faith, relev, correct = judge(q["query"], got_docs, answer, q["gold"])
  if correct < 1:
    print('\n>>> correctness problem:', correct, ', kind:', q["kind"],  
      '\n query:', q["query"], 
      '\n gold:', q["gold"],
      '\n docs:', q['relevant_docs'],
    )
  row = {
      "kind": q["kind"],                # Тип запроса для разбивки по срезам
      "query": q["query"],
      # Метрики поиска считаем по разделам-источникам найденных чанков
      # Метрики генерации: их проставил судья
      "faithfulness": faith,
      "answer relevance": relev,
      "correctness": correct,
      "answer": answer,                 
      "gold": q['gold'],                 
      "ctx": ctx,                       
      "doc_ids": doc_ids,                       
      "docs": got_docs,                       
  }
  rows.append(row)

  with open("6_judge_golden_log.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(row, ensure_ascii=False) + "\n")


>>> correctness problem: 0 , kind: semantic 
 query: скачанная серия перестала открываться без интернета 
 gold: истёк срок действия загрузки (30 дней); нужно подключиться к интернету для продления или скачать заново 
 docs: [49]

>>> correctness problem: 0 , kind: semantic 
 query: подключил колонки, но объёмного звука нет — идёт обычный стерео 
 gold: нужен тариф «Максимум», дорожка 5.1 у фильма и подключение по HDMI ARC или оптике; на динамиках телевизора звук сводится в стерео 
 docs: [22]

>>> correctness problem: 0 , kind: multi 
 query: чем Стандарт отличается от Максимума и потеряю ли оплаченные дни при переходе 
 gold: «Максимум» дороже и даёт 4K HDR, звук 5.1 и четыре одновременных просмотра; при переходе вверх неиспользованные дни пересчитываются, они не сгорают 
 docs: [113, 128]


In [68]:
'''
6.9.1 Golden Run Stat
'''
report = pd.DataFrame(rows)
METRICS = ["faithfulness", "answer relevance", "correctness"]

print("Total metrics - all queries:")
print(report[METRICS].mean().round(3))          # Средние по всему набору

print("\nПо типам запросов:")
print(report.groupby("kind")[METRICS].mean().round(3))   # Срезы по полю kind

Total metrics - all queries:
faithfulness        4.955
answer relevance    4.288
correctness         0.955
dtype: float64

По типам запросов:
          faithfulness  answer relevance  correctness
kind                                                 
exact            5.000             5.000        1.000
gap              5.000             1.000        1.000
multi            4.812             4.812        0.938
semantic         5.000             4.840        0.920


# Этап 6. Оценка качества и диагностика - Report

## Project Environment
Проект выполнялся локально на модели Qwen3.5-9B с кватизацией Q4_K_XL.
Модель запускалась на MacBook M4 16gb с использованием llama.cpp (native ru).

Infra: local, MacBook M4 16gb
Env: llama.cpp (native run)
Model/generation: unsloth/Qwen3.5-9B-GGUF Qwen3.5-9B-UD-Q4_K_XL.gguf
Model/reranking:  BAAI/bge-reranker-v2-m3
Model/embedding:  intfloat/multilingual-e5-base

После перехода с Docker'а на нативной запуск llama.cpp система стала работать стабильно. 
Однако, время обработки запросов - достаточно большое. Например, общее время прогона Судьи по 66 запросам - 85 мин.


## Матрица диагнозов

   kind   | precision |  recall@5 |   MRR  |faithfulness   | relevance  | correctness | manual check
 :-------:|:---------:|:---------:|:------:|:-------------:|:----------:|:-----------:|:--------------:
 exact    |   0.293   |   1.000   |  1.000 |   5.000       |   4.733    |    0.933    |blurred context
 semantic |   0.240   |   0.960   |  0.708 |   5.000       |   4.480 !  |    0.800 !  |blurred context
 multi    |   0.475   |   0.875 ! |  0.859 |   4.688 !     |   4.938    |    0.875    |iterative RAG 
 gap      |   0       |   0       |  0     |   5.000       |   1.400    |    1.000    |


## Ручной разбор
Приведены параметры, посчитанные на основе ручного разбора 8-ми проблемных запросов.

- Статистика по типам запросов
   kind   | к-во 
 :-------:|:-------
 exact    |   1   
 semantic |   5 
 multi    |   2 
 gap      |   0       

Проблема 1-го запроса exact предположительно связана с размытым (большим) контекстом.


- Статистика по типам проблем
тип проблемы | % от проблем (8 запросов)
-------------|--------------------------
 retrieval   |  0.188  - 1.5 из 8
 reranking   |  0.375  - 3   из 8
 generation  |  0.5    - 4   из 8


## Результат 

All 66 queries:
kind      precision@5  recall@5    MRR                              
exact           0.293     1.000  1.000
multi           0.475     0.875  0.859
semantic        0.240     0.960  0.708
gaps            0         0      0

queries (except gaps - 56 queries):
precision@5    |   0.321
recall@5       |   0.946
MRR            |   0.829

**
Формальные метрики:
- recall@5:    > 0.85 (0.946)
- correctness:  ~0.9  (0.879)
**


## Выводы

1. RAG
RAG работает достаточно хорошо - retrieval + re-ranking. 
Достигнут результат recall@5: 0.946.
precision (0.321) и MRR (0.829) - близко к максимальным значениям.

2. Обработка запросов

- Система хорошо реагирует на запросы типа exact.
Единственная проблема, идентифицированная судьей - некорректный ответ модели, который, видимо, связан с отсутствием специфичного термина HDMI ARC в выдаче.

- Запросы типа gap хорошо идентифицируются и отрабатываются на уровне модели.

- По запросам типа semantic и multi - поведение приемлемое. 
Улучшение возможно - как на уровне генерации, так и на уровне реранкинга.

В проблемных запросах - 3 проблемы реранкинга и 4 проблемы генерации.
2 из проблем генерации повторились при "золотом прогоне".

3. "Золотой прогон" 
"Золотой прогон" показал проблемы faithfulness для запросов типа multi, что говорит о потолке улучшения выдачи для запросов данного типа. Дальнейшее улучшение возможно путем разработки итеративной выдачи контекста.

**
Можно сделать вывод, что система готова к пилоту.
В то же время, система открыта к улучшениям и развитию. Возможные улучшения перечислены в разделе Рекомендации.
**

## Рекомендации

1. Переход на более сильную модель - Qwen3.5-9B с меньшей квантизацией, или на модель, с большим набором параметров.
   Ожидание - более сильная модель улучшит генерацию на размытом контексте.
2. Для улучшения реранкинга можно рассмотреть использование более сильной модели, чем используемая BAAI/bge-reranker-v2-m3.
   Ожидание - более сильная модель улучшит реранкинг на размытом контексте.
3. Для улучшения ответов по запросам типа multi рекомендуется разработки итеративного формирования контекста.
   Ожидание - улучшение выдачи по запросам типа multi.